# DATA PREPARATION SCRIPTS - CHIPATA CITY COUNCIL DATASETS

## A. BUDGET ALLOCATION SCRIPTS

## Setup
Installing `pymupdf` for extracting text directly from the budget PDF.

In [ ]:
!pip install pymupdf -q

## Load source documents
Uploading the Chipata City Council 2024 Output-Based Annual Budget and the
2025 Bi-annual Budget Performance report — the two source PDFs for the
budget allocations table.

In [ ]:
from google.colab import files
uploaded = files.upload()

## Confirm uploaded filenames
Checking the exact filenames Colab saved, since upload can alter naming
(e.g. appended characters) — needed to reference the files correctly in code.

In [ ]:
import os
print(os.listdir())

## Standardise filename
Renaming the budget PDF to a simpler, consistent name for use throughout
the notebook, since the uploaded filename may include extra characters.

In [ ]:
files_here = [f for f in os.listdir() if 'Chipata' in f and 'budget' in f]
if files_here:
    os.rename(files_here[0], "budget_2024.pdf")
print(os.listdir())

## Inspect document structure
Opening the budget PDF and extracting text from page 1 to confirm the file
has a native text layer (not scanned) and to understand its structure
before writing extraction logic.

In [ ]:
import pymupdf

doc = pymupdf.open("budget_2024.pdf")
print(f"Total pages: {len(doc)}")
print("--- PAGE 1 TEXT ---")
print(doc[0].get_text())

## Locate the budget tables
Scanning all pages to identify where the actual financial data sits.
Pages 1–2 contain mandate/strategy narrative; pages 3–4 cover revenue
(handled separately in the revenue_streams table); page 8 contains the
master Budget Allocation by Programme and Sub-Programme table (Table 3);
pages 11 onward break this down per individual programme (Constituency
Development, Local Governance, IDP, etc.) with Approved figures for
2022, 2023, and 2024.

In [ ]:
for i in range(len(doc)):
    text = doc[i].get_text()
    print(f"=== Page {i+1} ({len(text)} chars) ===")
    print(text[:200])
    print()

## Extract full text from key tables
Pulling the complete text of page 8 (master Budget Allocation by Programme
and Sub-Programme summary) and page 11 (first individual programme
breakdown — Constituency Development) to examine how figures, years, and
programme names are laid out before writing the parser.

In [ ]:
print("=== PAGE 8 FULL TEXT ===")
print(doc[7].get_text())  # index 7 = page 8

print("\n\n=== PAGE 11 FULL TEXT ===")
print(doc[10].get_text())  # index 10 = page 11

## Parse Table 3 — Budget Allocation by Programme and Sub-Programme (page 8)
The PDF's table structure extracts as flat text with columns out of visual order
(a known pymupdf limitation on multi-row-header tables). Each programme/sub-programme
is a fixed 7-line block: name, code, then 5 numeric values. In this document, only
one value per row is a real figure — the rest are placeholders ("(0)" or "-") since
prior-year comparative data isn't populated in this budget cycle. We extract that
single real value as the 2024 approved budget amount. Rows are flagged
`needs_review` since the 2022/2023 column mapping isn't reliably determined from
text order alone — this will need spot-checking against the source PDF.

In [ ]:
import pandas as pd

page8_text = doc[7].get_text()
lines = [l.strip() for l in page8_text.split('\n') if l.strip()]

# Data starts right after the header line "Estimate"
start_idx = lines.index('Estimate') + 1
data_lines = lines[start_idx:]

rows = []
i = 0
while i <= len(data_lines) - 7:
    name = data_lines[i]
    code = data_lines[i+1]

    # sanity check: code should be a short number (1-3 digits)
    if not code.replace('.', '').isdigit() or len(code) > 3:
        i += 1
        continue

    raw_values = data_lines[i+2:i+7]
    numeric_values = []
    for v in raw_values:
        v_clean = v.replace(',', '').replace('(0)', '0').replace('-', '0').strip()
        try:
            numeric_values.append(float(v_clean))
        except ValueError:
            numeric_values.append(0.0)

    approved_2024 = max(numeric_values) if numeric_values else 0.0

    rows.append({
        'record_id': f"BUD-2024-{code}",
        'council_name': 'Chipata City Council',
        'budget_year': 2024,
        'budget_period': 'annual',
        'vote_head_code': code,
        'programme_name': name,
        'sub_programme': '',
        'budget_category': 'other',
        'approved_budget_zmw': approved_2024,
        'source_document': 'budget_2024.pdf',
        'source_page': 8,
        'extraction_method': 'native_text',
        'data_quality_flag': 'needs_review',
        'notes': 'Column order inferred from single non-zero value per row; verify against source PDF'
    })
    i += 7

df_page8 = pd.DataFrame(rows)
print(f"Extracted {len(df_page8)} rows")
df_page8

## Fix duplicate record IDs and mark programme vs sub-programme
Vote head codes reset within each parent programme (e.g. code "001" appears
under multiple different programmes), so `record_id` built from code alone
isn't unique. Rebuilding it using a running row index instead. Also flagging
whether each row is a top-level programme (single-digit code, e.g. "1", "2")
or a sub-programme (3-digit code, e.g. "779", "044") based on code length,
since the source table nests sub-programmes under their parent programme.

In [ ]:
df_page8['record_id'] = [f"BUD-2024-{i:03d}" for i in range(1, len(df_page8) + 1)]

df_page8['programme_level'] = df_page8['vote_head_code'].apply(
    lambda c: 'programme' if len(c.strip()) <= 2 else 'sub_programme'
)

# quick sanity check
print(df_page8[['record_id', 'vote_head_code', 'programme_name', 'programme_level', 'approved_budget_zmw']].to_string())

## Check for LGEF-specific content
The assignment brief explicitly requires "LGEF utilisation" data. Searching
the full document for any mentions of LGEF (Local Government Equalisation
Fund) to confirm whether it appears as a distinct budget line or funding
source that needs separate tagging.

In [ ]:
for i in range(len(doc)):
    text = doc[i].get_text()
    if 'LGEF' in text.upper() or 'EQUALISATION' in text.upper():
        print(f"Page {i+1} mentions LGEF:")
        idx = text.upper().find('LGEF') if 'LGEF' in text.upper() else text.upper().find('EQUALISATION')
        print(text[max(0,idx-100):idx+300])
        print("---")

## LGEF finding
Page 4 shows LGEF as a revenue line item (K15.2M approved for 2024) under
"Grants" — this is the council's LGEF *income*, which belongs in the revenue
streams table, not budget allocations.
Page 8's programme-level budget table doesn't break down funding source per
programme, so a true "LGEF utilisation by programme" breakdown isn't
extractable from this document. Flagging this gap rather than fabricating
a mapping — noting it in the Data Description Paper's limitations section.

## Load and inspect bi-annual performance report
Opening the second source document to check its structure before extracting
actual expenditure figures, which will let us calculate budget variance
per programme.

In [ ]:
doc2 = pymupdf.open("2025-Bi-annual-Budget-Perfomance.pdf")
print(f"Total pages: {len(doc2)}")
for i in range(min(5, len(doc2))):
    print(f"=== Page {i+1} ===")
    print(doc2[i].get_text()[:400])
    print()

## Confirm scan status — OCR required
Native text extraction on the bi-annual performance PDF returns garbled/unreadable
characters, indicating this file is scanned (image-based), not text-native,
despite earlier assumptions. OCR is required to extract this document.

## Set up OCR pipeline for bi-annual performance report
The bi-annual budget performance PDF is scanned/image-based, so native text
extraction fails. Installing Tesseract OCR and pdf2image to convert each
page to an image and extract text via OCR instead. This will let us pull
actual expenditure figures to calculate budget variance per programme,
strengthening the "LGEF utilisation" and comprehensiveness requirements
in the assignment brief.

In [ ]:
!apt-get install -y tesseract-ocr poppler-utils -q
!pip install pytesseract pdf2image -q

## Convert scanned pages to images and run OCR
Converting each page of the bi-annual performance PDF to a high-resolution
image, then extracting text via Tesseract OCR. Starting with page 1 only
to verify OCR quality before running it across all 8 pages.

In [ ]:
from pdf2image import convert_from_path
import pytesseract

images = convert_from_path("2025-Bi-annual-Budget-Perfomance.pdf", dpi=300, first_page=1, last_page=1)
ocr_text = pytesseract.image_to_string(images[0])
print(ocr_text)

## Visually inspect the scanned page
OCR output on page 1 was mostly garbled. Displaying the raw converted image
to check for rotation, skew, or resolution issues before adjusting the OCR
pipeline.

In [ ]:
images[0]

## Rotate image and re-run OCR
Page was scanned rotated (confirmed via visual inspection), which degrades
OCR accuracy significantly even on a clear, high-quality scan. Rotating
90° counter-clockwise corrects the orientation before extracting text.

In [ ]:
# Rotate 90° counter-clockwise to correct the scan orientation
rotated = images[0].rotate(-90, expand=True)
rotated

In [ ]:
ocr_text = pytesseract.image_to_string(rotated)
print(ocr_text)

## OCR successful — check document scope
Rotation fix resolved OCR quality significantly. However, this page shows
revenue performance (budget vs. actual receipts by revenue source: Local
Taxes, Fees & Charges, Grants), not programme-level expenditure. This maps
more directly to the revenue_streams table than budget_allocations.
Checking remaining pages before deciding final placement.

In [ ]:
# Process all 8 pages now that rotation angle is confirmed
all_images = convert_from_path("2025-Bi-annual-Budget-Perfomance.pdf", dpi=300)

for i, img in enumerate(all_images):
    rotated_img = img.rotate(-90, expand=True)
    text = pytesseract.image_to_string(rotated_img)
    print(f"=== PAGE {i+1} ===")
    print(text[:400])
    print()

## Fix programme/sub-programme structure and infer budget category
Restructuring so `programme_name` always holds the parent programme (e.g.
"Constituency Development"), and `sub_programme` holds the child line item
name (e.g. "Community Projects") where applicable — matching the schema's
intended structure. Also inferring `budget_category` from programme context:
CDF-related programmes are tagged `cdf`, core administrative/support
programmes as `recurrent_expenditure`, and infrastructure/asset-heavy
programmes as `capital_expenditure` where the source text confirms this
(e.g. page 6 states Housing and Community Amenities is largely capital
assets). Rows without a clear signal are left as `other` and flagged for
manual review.

In [ ]:
# Step 1: forward-fill parent programme name into sub-programme rows
current_parent = None
fixed_programme_name = []
fixed_sub_programme = []

for _, row in df_page8.iterrows():
    if row['programme_level'] == 'programme':
        current_parent = row['programme_name']
        fixed_programme_name.append(row['programme_name'])
        fixed_sub_programme.append('')
    else:  # sub_programme
        fixed_programme_name.append(current_parent)
        fixed_sub_programme.append(row['programme_name'])

df_page8['programme_name'] = fixed_programme_name
df_page8['sub_programme'] = fixed_sub_programme

# Step 2: infer budget_category based on parent programme context
category_map = {
    'Constituency Development': 'cdf',
    'Management and support  Services': 'recurrent_expenditure',
    'Resource Mobilisation and Management': 'recurrent_expenditure',
    'Housing and Community Amenities': 'capital_expenditure',
    'Local Governance': 'recurrent_expenditure',
    'District Health servcies': 'recurrent_expenditure',
}

df_page8['budget_category'] = df_page8['programme_name'].map(category_map).fillna('other')

# quick check
print(df_page8[['programme_name', 'sub_programme', 'vote_head_code', 'budget_category', 'approved_budget_zmw']].to_string())

## Validate total against stated budget figure
Cross-checking the sum of top-level programme approved amounts against the
K150.2 million total stated in the narrative (page 5), to confirm whether
page 8's table was fully captured or continues beyond what was extracted.

In [ ]:
programme_total = df_page8[df_page8['sub_programme'] == '']['approved_budget_zmw'].sum()
print(f"Sum of top-level programmes: {programme_total:,.2f} ZMW")
print(f"Stated total in document: ~150,200,000 ZMW")
print(f"Difference: {150_200_000 - programme_total:,.2f} ZMW")

## Check for table continuation on page 9
Page 8's extracted text ends abruptly after "Primary Health Services,"
suggesting the table may continue onto page 9 with additional programmes
or sub-programmes not yet captured. Checking page 9's full text to confirm.

In [ ]:
print(doc[8].get_text())  # index 8 = page 9

## Extract remaining rows from page 9 and merge
Page 8's table continues onto page 9 after the narrative paragraph, with
3 additional programmes/sub-programmes (District Health Coordination,
Veterinary services, Transport Services) not yet captured. The page also
confirms a Head Total of 150,164,034 ZMW, matching the narrative figure
and validating completeness once these rows are added. Manually adding
these since the table fragment is small and mixed with narrative text
(same parsing logic as page 8 would misfire on this fragment).

In [ ]:
extra_rows = [
    {'record_id': None, 'council_name': 'Chipata City Council', 'budget_year': 2024,
     'budget_period': 'annual', 'vote_head_code': '002', 'programme_name': 'District Health servcies',
     'sub_programme': 'District Health Coordination', 'budget_category': 'recurrent_expenditure',
     'approved_budget_zmw': 3968017.0, 'source_document': 'budget_2024.pdf', 'source_page': 9,
     'extraction_method': 'native_text', 'data_quality_flag': 'needs_review',
     'notes': 'Continuation of page 8 table, extracted from page 9'},
    {'record_id': None, 'council_name': 'Chipata City Council', 'budget_year': 2024,
     'budget_period': 'annual', 'vote_head_code': '14', 'programme_name': 'Veterinary services',
     'sub_programme': '', 'budget_category': 'other',
     'approved_budget_zmw': 189173.0, 'source_document': 'budget_2024.pdf', 'source_page': 9,
     'extraction_method': 'native_text', 'data_quality_flag': 'needs_review',
     'notes': 'Continuation of page 8 table, extracted from page 9'},
    {'record_id': None, 'council_name': 'Chipata City Council', 'budget_year': 2024,
     'budget_period': 'annual', 'vote_head_code': '001', 'programme_name': 'Veterinary services',
     'sub_programme': 'Animal Health Extension Services', 'budget_category': 'other',
     'approved_budget_zmw': 189173.0, 'source_document': 'budget_2024.pdf', 'source_page': 9,
     'extraction_method': 'native_text', 'data_quality_flag': 'needs_review',
     'notes': 'Continuation of page 8 table, extracted from page 9'},
    {'record_id': None, 'council_name': 'Chipata City Council', 'budget_year': 2024,
     'budget_period': 'annual', 'vote_head_code': '15', 'programme_name': 'Transport Services',
     'sub_programme': '', 'budget_category': 'other',
     'approved_budget_zmw': 3708668.0, 'source_document': 'budget_2024.pdf', 'source_page': 9,
     'extraction_method': 'native_text', 'data_quality_flag': 'needs_review',
     'notes': 'Continuation of page 8 table, extracted from page 9'},
    {'record_id': None, 'council_name': 'Chipata City Council', 'budget_year': 2024,
     'budget_period': 'annual', 'vote_head_code': '001', 'programme_name': 'Transport Services',
     'sub_programme': 'Road Transport', 'budget_category': 'other',
     'approved_budget_zmw': 3708668.0, 'source_document': 'budget_2024.pdf', 'source_page': 9,
     'extraction_method': 'native_text', 'data_quality_flag': 'needs_review',
     'notes': 'Continuation of page 8 table, extracted from page 9'},
]

import pandas as pd
df_page8 = pd.concat([df_page8, pd.DataFrame(extra_rows)], ignore_index=True)

# rebuild record_id sequentially for all rows
df_page8['record_id'] = [f"BUD-2024-{i:03d}" for i in range(1, len(df_page8) + 1)]

# re-validate total
top_level_total = df_page8[df_page8['sub_programme'] == '']['approved_budget_zmw'].sum()
print(f"New top-level total: {top_level_total:,.2f} ZMW")
print(f"Head Total stated in document: 150,164,034.00 ZMW")
print(f"Difference: {150_164_034 - top_level_total:,.2f} ZMW")
print(f"\nTotal rows now: {len(df_page8)}")

## Improve OCR quality on the Payments page (page 2)
Re-converting page 2 at higher resolution (400 DPI) with grayscale and
contrast enhancement to improve OCR accuracy on the scanned Payments table,
since the initial pass at standard resolution produced too many blanks
and garbled values to use reliably.

In [ ]:
from PIL import ImageEnhance

# Re-convert just page 2 at higher DPI
page2_images = convert_from_path("2025-Bi-annual-Budget-Perfomance.pdf", dpi=400, first_page=2, last_page=2)
page2_rotated = page2_images[0].rotate(-90, expand=True)

# Preprocess: grayscale + boost contrast
page2_gray = page2_rotated.convert('L')
enhancer = ImageEnhance.Contrast(page2_gray)
page2_enhanced = enhancer.enhance(2.0)

page2_enhanced

## Extract text via OCR with table-optimized settings
Running Tesseract with `--psm 6` (assumes a uniform block of text, better
suited to tabular layouts than the default page-segmentation mode).

In [ ]:
ocr_text_page2 = pytesseract.image_to_string(page2_enhanced, config='--psm 6')
print(ocr_text_page2)

## Crop table region and retry OCR
Full-page OCR remains unreliable on individual payment line items due to
scan quality and dense table formatting. Cropping to isolate just the
table body (excluding headers/footers) and retrying at higher resolution
with column-oriented PSM mode, to see if this improves accuracy before
deciding whether line-item detail is extractable at all.

In [ ]:
# View full image with dimensions to identify crop coordinates
width, height = page2_enhanced.size
print(f"Image size: {width} x {height}")
page2_enhanced

## Crop to isolate table region
Cropping the image to the estimated table boundaries (excluding header
and footer/signature areas) to reduce noise and improve OCR accuracy.

In [ ]:
width, height = page2_enhanced.size

# Adjust these fractions based on where the table actually sits
top_frac = 0.20      # <- replace with your estimate
bottom_frac = 0.75   # <- replace with your estimate

top = int(height * top_frac)
bottom = int(height * bottom_frac)

cropped = page2_enhanced.crop((0, top, width, bottom))
cropped

## Transcribe Payments table from page 2 (visual verification)
OCR on this scan remained unreliable even after rotation, contrast
enhancement, and cropping attempts. The table image is clearly legible on
visual inspection, so values are transcribed directly rather than forcing
unreliable automated extraction. This is documented as `extraction_method
= manual` with `data_quality_flag = verified`, since figures were read
directly from the source rather than inferred or OCR'd. This table uses
economic classification (Personal emoluments, Goods & services, etc.)
rather than programme classification, covering the 2025 bi-annual period
(01-Jan-2025 to 30-Jun-2025), budgeted against a full-year 2025 approved
figure.

In [ ]:
payments_rows = [
    {'sub_programme': 'Personal emoluments', 'budget_category': 'recurrent_expenditure',
     'approved_budget_zmw': 42622728.00, 'actual_expenditure_zmw': 15581875.77, 'variance_zmw': 27040852.23, 'variance_percent': 37.0},
    {'sub_programme': 'Use of goods and services', 'budget_category': 'recurrent_expenditure',
     'approved_budget_zmw': 38064123.00, 'actual_expenditure_zmw': 16889204.68, 'variance_zmw': 21174918.32, 'variance_percent': 44.0},
    {'sub_programme': 'Financial charges', 'budget_category': 'debt_service',
     'approved_budget_zmw': 0.0, 'actual_expenditure_zmw': 0.0, 'variance_zmw': 0.0, 'variance_percent': None},
    {'sub_programme': 'Social benefits', 'budget_category': 'other',
     'approved_budget_zmw': 48334175.00, 'actual_expenditure_zmw': 27800800.00, 'variance_zmw': 20533375.00, 'variance_percent': 58.0},
    {'sub_programme': 'Non-financial assets', 'budget_category': 'capital_expenditure',
     'approved_budget_zmw': 52852445.00, 'actual_expenditure_zmw': 32289463.15, 'variance_zmw': 20562981.85, 'variance_percent': 61.0},
    {'sub_programme': 'Financial assets', 'budget_category': 'other',
     'approved_budget_zmw': 7473872.00, 'actual_expenditure_zmw': 326600.00, 'variance_zmw': 7147272.00, 'variance_percent': 4.0},
    {'sub_programme': 'Loan repayments', 'budget_category': 'debt_service',
     'approved_budget_zmw': 0.0, 'actual_expenditure_zmw': 0.0, 'variance_zmw': 0.0, 'variance_percent': None},
    {'sub_programme': 'Other repayments', 'budget_category': 'other',
     'approved_budget_zmw': 3478550.00, 'actual_expenditure_zmw': 3135845.14, 'variance_zmw': 342704.86, 'variance_percent': 90.0},
]

for row in payments_rows:
    row.update({
        'council_name': 'Chipata City Council',
        'budget_year': 2025,
        'budget_period': 'bi_annual',
        'vote_head_code': '',
        'programme_name': 'Total Council (Economic Classification)',
        'source_document': '2025-Bi-annual-Budget-Perfomance.pdf',
        'source_page': 2,
        'extraction_method': 'manual',
        'data_quality_flag': 'verified',
        'notes': 'Economic classification breakdown, transcribed visually from scanned Payments table (OCR unreliable)'
    })

df_payments = pd.DataFrame(payments_rows)
df_page8 = pd.concat([df_page8, df_payments], ignore_index=True)
df_page8['record_id'] = [f"BUD-{i:03d}" for i in range(1, len(df_page8) + 1)]

# verify total payments matches source
total_check = df_payments['approved_budget_zmw'].sum()
print(f"Sum of transcribed payment categories: {total_check:,.2f}")
print(f"Total payments stated in document: 192,825,893.00")
print(f"Match: {abs(total_check - 192825893.00) < 1}")
print(f"\nTotal rows in budget_allocations now: {len(df_page8)}")

## Validate required columns before export
Checking that every row has a value in all schema-required columns
(record_id, council_name, budget_year, budget_period, programme_name,
budget_category, source_document, extraction_method, extracted_at) before
final export, to catch any gaps while they're still easy to fix.

In [ ]:
required_columns = ['record_id', 'council_name', 'budget_year', 'budget_period',
                     'programme_name', 'budget_category', 'source_document',
                     'extraction_method', 'extracted_at']

print("Checking required columns for missing/blank values...\n")

all_clear = True
for col in required_columns:
    missing = df_page8[df_page8[col].isna() | (df_page8[col].astype(str).str.strip() == '')]
    if len(missing) > 0:
        all_clear = False
        print(f"⚠️  '{col}' has {len(missing)} missing/blank row(s):")
        print(missing[['record_id', 'programme_name', 'sub_programme']].to_string())
        print()
    else:
        print(f"✅ '{col}' — no missing values")

print(f"\n{'✅ All required columns complete.' if all_clear else '⚠️ Some required columns need fixing before export.'}")

## Add extraction timestamp and re-validate
Populating `extracted_at` (the last required column) before final export,
then re-running the completeness check to confirm all required fields
are filled across all 60 rows.

In [ ]:
from datetime import datetime

df_page8['extracted_at'] = datetime.now().isoformat()

# re-run the check now that extracted_at exists
required_columns = ['record_id', 'council_name', 'budget_year', 'budget_period',
                     'programme_name', 'budget_category', 'source_document',
                     'extraction_method', 'extracted_at']

all_clear = True
for col in required_columns:
    missing = df_page8[df_page8[col].isna() | (df_page8[col].astype(str).str.strip() == '')]
    if len(missing) > 0:
        all_clear = False
        print(f"⚠️  '{col}' has {len(missing)} missing/blank row(s)")
    else:
        print(f"✅ '{col}' — no missing values")

print(f"\n{'✅ All required columns complete. Ready to export.' if all_clear else '⚠️ Fix issues before exporting.'}")

## Export final budget_allocations table
Saving the complete, validated table (60 rows, covering approved 2024
programme budgets and 2025 bi-annual actual payments by economic
classification) as the required pipe-separated CSV, following the naming
convention specified in the assignment brief.

In [ ]:
# ensure column order matches schema, keeping data_quality_flag as a documented extra
schema_columns = ['record_id', 'council_name', 'budget_year', 'budget_period', 'vote_head_code',
                   'programme_name', 'sub_programme', 'sector_cluster', 'budget_category', 'funding_source',
                   'approved_budget_zmw', 'revised_budget_zmw', 'actual_expenditure_zmw', 'variance_zmw',
                   'variance_percent', 'source_document', 'source_page', 'extraction_method',
                   'extracted_at', 'notes']

for col in schema_columns:
    if col not in df_page8.columns:
        df_page8[col] = ''

df_final = df_page8[schema_columns + ['data_quality_flag']]

df_final.to_csv("db-unza26-csc4792-chipata_budget_allocations.csv", sep="|", index=False, encoding="utf-8")
print(f"Exported {len(df_final)} rows.")
print(os.listdir())

## Verify export integrity
Reading the exported CSV back in to confirm the pipe separator and encoding
work correctly and no data was lost or corrupted during export.

In [ ]:
df_check = pd.read_csv("db-unza26-csc4792-chipata_budget_allocations.csv", sep="|")
print(f"Rows read back: {len(df_check)}")
print(f"Columns: {list(df_check.columns)}")
df_check.head()

In [ ]:
from google.colab import files
files.download("db-unza26-csc4792-chipata_budget_allocations.csv")

## Web scraping — Chipata City Council website
Visiting the council's official website to fulfil the assignment's explicit
requirement to extract data "from the respective council websites." Checking
for budget/financial documents that may be available online, in addition to
the PDFs already sourced locally.

In [ ]:
!pip install requests beautifulsoup4 -q

import requests
from bs4 import BeautifulSoup

url = "https://www.chipatacouncil.gov.zm"
response = requests.get(url, timeout=15)
print(f"Status code: {response.status_code}")

soup = BeautifulSoup(response.content, 'html.parser')
print(f"Page title: {soup.title.string if soup.title else 'No title found'}")

# List all links on the homepage
links = soup.find_all('a', href=True)
print(f"\nTotal links found: {len(links)}")
for link in links[:30]:
    print(link['href'], "|", link.get_text(strip=True)[:60])

## Web scraping — Chipata City Council website
Visiting the council's official website to fulfil the assignment's explicit
requirement to extract data "from the respective council websites." The
site returned an SSL certificate verification error (common with
government sites using outdated/misconfigured certificates) — verification
is disabled for this read-only request since no sensitive data is being
transmitted.

In [ ]:
import requests
from bs4 import BeautifulSoup
import warnings
warnings.filterwarnings('ignore')

url = "https://www.chipatacouncil.gov.zm"
response = requests.get(url, timeout=15, verify=False)
print(f"Status code: {response.status_code}")

soup = BeautifulSoup(response.content, 'html.parser')
print(f"Page title: {soup.title.string if soup.title else 'No title found'}")

links = soup.find_all('a', href=True)
print(f"\nTotal links found: {len(links)}")
for link in links[:30]:
    print(link['href'], "|", link.get_text(strip=True)[:60])

## Explore Finance Department and Publications pages
Two links from the homepage appear directly relevant to budget data: the
Finance Department page and the Publications page (often where councils
post budget/financial PDF documents). Visiting both to check for content
or downloadable files relevant to budget_allocations.

In [ ]:
finance_url = "https://www.chipatacouncil.gov.zm/?page_id=5692"
pubs_url = "https://www.chipatacouncil.gov.zm/?page_id=541"

for label, page_url in [("FINANCE DEPARTMENT", finance_url), ("PUBLICATIONS", pubs_url)]:
    print(f"=== {label} ({page_url}) ===")
    resp = requests.get(page_url, timeout=15, verify=False)
    page_soup = BeautifulSoup(resp.content, 'html.parser')

    # print main visible text (rough cut, first 1000 chars)
    text = page_soup.get_text(separator=' ', strip=True)
    print(text[:1000])

    # look specifically for PDF links
    pdf_links = [a['href'] for a in page_soup.find_all('a', href=True) if '.pdf' in a['href'].lower()]
    print(f"\nPDF links found: {len(pdf_links)}")
    for pdf in pdf_links:
        print(" -", pdf)
    print("\n")

## Confirm live source and check for newer documents
Cross-referencing the council's Publications page against the PDFs already
processed confirms they are the same files hosted live on the official
website (Chipata-City-Council-budget-2024.pdf, 2025-Bi-annual-Budget-Perfomance.pdf),
satisfying the requirement to extract data from the council's website
directly. The scrape also surfaced a document not previously on hand —
budget-approved-2026.pdf — downloaded here to check whether it extends
budget coverage into a third year.

In [ ]:
new_budget_url = "https://www.chipatacouncil.gov.zm/wp-content/uploads/2026/05/budget-approved-2026.pdf"

resp = requests.get(new_budget_url, timeout=20, verify=False)
with open("budget_2026.pdf", "wb") as f:
    f.write(resp.content)

print(f"Downloaded: {len(resp.content)} bytes")

doc2026 = pymupdf.open("budget_2026.pdf")
print(f"Total pages: {len(doc2026)}")
print("--- PAGE 1 TEXT ---")
print(doc2026[0].get_text())

In [ ]:
try:
    print(f"df_page8 exists with {len(df_page8)} rows")
except NameError:
    print("df_page8 does NOT exist — needs rebuilding")

try:
    print(f"doc2026 exists with {len(doc2026)} pages")
except NameError:
    print("doc2026 does NOT exist — needs rebuilding")

In [ ]:
import pymupdf

new_budget_url = "https://www.chipatacouncil.gov.zm/wp-content/uploads/2026/05/budget-approved-2026.pdf"

resp = requests.get(new_budget_url, timeout=20, verify=False)
with open("budget_2026.pdf", "wb") as f:
    f.write(resp.content)

print(f"Downloaded: {len(resp.content)} bytes")

doc2026 = pymupdf.open("budget_2026.pdf")
print(f"Total pages: {len(doc2026)}")
print("--- PAGE 1 TEXT ---")
print(doc2026[0].get_text())

## Check scan status of 2026 budget
Page 1 returned no extractable text, and the file size (38MB for 57 pages)
is unusually large for a text-native PDF — both signs point to this being
a scanned document. Checking a few more pages to confirm before deciding
whether to pursue OCR/manual transcription or treat this as a
lower-priority addition given the size.

In [ ]:
for i in range(min(5, len(doc2026))):
    text = doc2026[i].get_text()
    print(f"=== Page {i+1} ({len(text)} chars) ===")
    print(text[:200] if text else "(no extractable text)")
    print()

## Quick check — locate summary table in 2026 budget
Before committing to full OCR across 57 pages, checking whether the 2026
budget follows the same structure as the 2024 version (programme summary
table typically appearing around page 8-9). Converting just a few candidate
pages to images and running OCR to confirm before deciding on full
extraction scope.

In [ ]:
from pdf2image import convert_from_path
import pytesseract

# Check pages 7-10 (covers the range where the table sat in the 2024 doc)
candidate_pages = convert_from_path("budget_2026.pdf", dpi=300, first_page=7, last_page=10)

for i, img in enumerate(candidate_pages):
    page_num = i + 7
    # try both orientations since we don't know if this scan is rotated
    text_normal = pytesseract.image_to_string(img)
    text_rotated = pytesseract.image_to_string(img.rotate(-90, expand=True))

    print(f"=== PAGE {page_num} (normal) ===")
    print(text_normal[:300])
    print(f"\n=== PAGE {page_num} (rotated -90) ===")
    print(text_rotated[:300])
    print("\n" + "="*50 + "\n")

## OCR the full 2026 programme budget table
Pages 8-10 confirm the same programme/sub-programme table structure as the
2024 document, with the 2024 approved figures matching exactly — confirming
this document extends the same table with 2025 and 2026 columns. Running
OCR across the full table range (pages 8-11) to capture all three years'
figures, extending the budget_allocations table with newer approved and
estimated budget data.

In [ ]:
table_pages = convert_from_path("budget_2026.pdf", dpi=400, first_page=8, last_page=12)

full_table_text = []
for i, img in enumerate(table_pages):
    text = pytesseract.image_to_string(img, config='--psm 6')
    full_table_text.append(text)
    print(f"=== PAGE {i+8} ===")
    print(text)
    print("\n")

## Extract 2025 and 2026 programme-level budgets
Table:2 on page 8 of the 2026 document is a clean summary table showing
approved budgets for 2024, 2025, and 2026 at the programme level. The 2024
column closely matches the figures already validated from the original
2024 document (minor variance likely reflects a later restatement in this
newer publication). The detailed sub-programme breakdown on pages 9-10 has
significant OCR degradation and is not extracted at this level of detail —
only the programme-level 2025 and 2026 figures are added, transcribed and
corrected from OCR against the visible table structure.

In [ ]:
# Programme-level data for 2025 and 2026, transcribed from Table:2 (page 8)
# Format: (vote_head_code, programme_name, budget_2025, budget_2026)
programme_2025_2026 = [
    ('1', 'Constituency Development', 72116301, 80065100),
    ('2', 'Local Governance', 3536881, 3773018),
    ('3', 'Integrated Development Planning', 6187954, 7173821),
    ('4', 'Economic and business development', 2110622, 2609043),
    ('5', 'Public health and Environmental protection', 4546772, 7847518),
    ('6', 'Housing and Community Amenities', 15687300, 17904197),
    ('7', 'Recreation Culture and Religion', 267068, 267069),
    ('8', 'Education and skills development', 130251, 130925),
    ('10', 'Public order and safety', 7182633, 6620906),
    ('11', 'Management and support  Services', 31284866, 31702297),
    ('12', 'Resource Mobilisation and Management', 4806840, 5535656),
    ('13', 'District Health servcies', 7790285, 8052415),
    ('15', 'Transport Services', 3200587, 6351350),
    ('16', 'Agricultural Services', 516367, 508368),
    ('17', 'Fisheries and Livestock', 486160, 503176),
    ('18', 'Social Protection and Community Development', 32975006, 13582709),
]

category_map_extended = {
    'Constituency Development': 'cdf',
    'Management and support  Services': 'recurrent_expenditure',
    'Resource Mobilisation and Management': 'recurrent_expenditure',
    'Housing and Community Amenities': 'capital_expenditure',
    'Local Governance': 'recurrent_expenditure',
    'District Health servcies': 'recurrent_expenditure',
}

new_rows = []
for code, name, amt_2025, amt_2026 in programme_2025_2026:
    for year, amt in [(2025, amt_2025), (2026, amt_2026)]:
        new_rows.append({
            'council_name': 'Chipata City Council',
            'budget_year': year,
            'budget_period': 'annual',
            'vote_head_code': code,
            'programme_name': name,
            'sub_programme': '',
            'budget_category': category_map_extended.get(name, 'other'),
            'approved_budget_zmw': amt,
            'source_document': 'budget_2026.pdf',
            'source_page': 8,
            'extraction_method': 'manual',
            'data_quality_flag': 'needs_review',
            'notes': 'Programme-level only; sub-programme detail not extracted due to OCR degradation on pages 9-10'
        })

df_new_years = pd.DataFrame(new_rows)
df_page8 = pd.concat([df_page8, df_new_years], ignore_index=True)
df_page8['record_id'] = [f"BUD-{i:03d}" for i in range(1, len(df_page8) + 1)]

print(f"Added {len(df_new_years)} new rows (2025 + 2026 programme budgets)")
print(f"Total rows now: {len(df_page8)}")

# validate against stated totals for each year
total_2025 = df_page8[(df_page8['budget_year']==2025) & (df_page8['sub_programme']=='')]['approved_budget_zmw'].sum()
total_2026 = df_page8[(df_page8['budget_year']==2026) & (df_page8['sub_programme']=='')]['approved_budget_zmw'].sum()
print(f"\n2025 total: {total_2025:,.2f} (document shows ~192,825,892 - note likely OCR misread first digit)")
print(f"2026 total: {total_2026:,.2f} (document Head Total: 192,627,565)")

## Cross-document validation
The 2025 programme-level total extracted from the 2026 budget document
(192,825,893.00) exactly matches the "Total payments" figure independently
transcribed from the 2025 bi-annual performance report — strong
cross-document validation. The 2026 total (192,627,568.00) is within 3 ZMW
of the document's own stated Head Total (192,627,565), consistent with
rounding across 16 programmes.

In [ ]:
from datetime import datetime
df_page8['extracted_at'] = datetime.now().isoformat()

required_columns = ['record_id', 'council_name', 'budget_year', 'budget_period',
                     'programme_name', 'budget_category', 'source_document',
                     'extraction_method', 'extracted_at']

all_clear = True
for col in required_columns:
    missing = df_page8[df_page8[col].isna() | (df_page8[col].astype(str).str.strip() == '')]
    if len(missing) > 0:
        all_clear = False
        print(f"⚠️  '{col}' has {len(missing)} missing/blank row(s)")
    else:
        print(f"✅ '{col}' — no missing values")

print(f"\n{'✅ All required columns complete. Ready to export.' if all_clear else '⚠️ Fix issues before exporting.'}")
print(f"Total rows: {len(df_page8)}")

## Export final budget_allocations table
Saving the complete, validated table (92 rows, covering approved programme
budgets for 2024, 2025, and 2026, plus actual 2025 payments by economic
classification) as the required pipe-separated CSV, following the naming
convention specified in the assignment brief.

In [ ]:
schema_columns = ['record_id', 'council_name', 'budget_year', 'budget_period', 'vote_head_code',
                   'programme_name', 'sub_programme', 'sector_cluster', 'budget_category', 'funding_source',
                   'approved_budget_zmw', 'revised_budget_zmw', 'actual_expenditure_zmw', 'variance_zmw',
                   'variance_percent', 'source_document', 'source_page', 'extraction_method',
                   'extracted_at', 'notes']

for col in schema_columns:
    if col not in df_page8.columns:
        df_page8[col] = ''

df_final = df_page8[schema_columns + ['data_quality_flag']]

df_final.to_csv("db-unza26-csc4792-chipata_budget_allocations.csv", sep="|", index=False, encoding="utf-8")
print(f"Exported {len(df_final)} rows.")
print(os.listdir())

In [ ]:
df_check = pd.read_csv("db-unza26-csc4792-chipata_budget_allocations.csv", sep="|")
print(f"Rows read back: {len(df_check)}")
print(f"Columns: {list(df_check.columns)}")
df_check.head()

In [ ]:
from google.colab import files
files.download("db-unza26-csc4792-chipata_budget_allocations.csv")

## Remove data_quality_flag column
Dropping the data_quality_flag column to match the official schema exactly,
consistent with the other tables in the dataset. Quality-related context
for flagged rows remains preserved in the notes column where applicable.

In [ ]:
# preserve any quality flag info into notes before dropping, so nothing is lost
df_final['notes'] = df_final.apply(
    lambda row: (row['notes'] + ' | ' if row['notes'] else '') + f"[{row['data_quality_flag']}]"
    if row['data_quality_flag'] not in ('', 'verified') else row['notes'],
    axis=1
)

df_final_clean = df_final.drop(columns=['data_quality_flag'])

df_final_clean.to_csv("db-unza26-csc4792-chipata_budget_allocations.csv", sep="|", index=False, encoding="utf-8")
print(f"Exported {len(df_final_clean)} rows with {len(df_final_clean.columns)} columns")
print(list(df_final_clean.columns))

In [ ]:
try:
    print(f"df_page8 rows: {len(df_page8)}")
except NameError:
    print("df_page8 does NOT exist — session needs rebuilding")

try:
    print(f"doc2026 pages: {len(doc2026)}")
except NameError:
    print("doc2026 does NOT exist")

import os
print("\nFiles:", os.listdir())

In [ ]:
from datetime import datetime
df_page8['extracted_at'] = datetime.now().isoformat()

required_columns = ['record_id', 'council_name', 'budget_year', 'budget_period',
                     'programme_name', 'budget_category', 'source_document',
                     'extraction_method', 'extracted_at']

all_clear = True
for col in required_columns:
    missing = df_page8[df_page8[col].isna() | (df_page8[col].astype(str).str.strip() == '')]
    if len(missing) > 0:
        all_clear = False
        print(f"⚠️  '{col}' has {len(missing)} missing/blank row(s)")
    else:
        print(f"✅ '{col}' — no missing values")

print(f"\n{'✅ All required columns complete. Ready to export.' if all_clear else '⚠️ Fix issues before exporting.'}")
print(f"Total rows: {len(df_page8)}")

In [ ]:
df_check = pd.read_csv("db-unza26-csc4792-chipata_budget_allocations.csv", sep="|")
print(f"Final check — Rows: {len(df_check)}, Columns: {len(df_check.columns)}")

from google.colab import files
files.download("db-unza26-csc4792-chipata_budget_allocations.csv")

## B. REVENUE STREAMS EXTRACTION SCRIPTS

# Chipata City Council Revenue Streams Extraction

**Output table:** `db-unza26-csc4792-chipata_revenue_streams.csv`  
**Source folder:** `Budgets & Financial Statement`  
**Council:** Chipata City Council

This notebook extracts locally generated revenue and related income streams from Chipata City Council financial statements and budget performance reports, then exports a pipe-separated CSV.

### Source documents used
| File | Year(s) | Content |
|------|---------|--------|
| `FINANCIAL-STATEMENT.pdf` | 2023 | Detailed revenue notes + budget vs actual variances |
| `FINANCIAL-STATEMENTS-FOR-THE-YEAR-ENDED-31TH.pdf` | 2022 | Detailed revenue notes |
| `1.-Chipata-City-Council-Bi-Annual-2025-Draft.pdf` | 2025 (Jan–Jun) | Semi-annual cash receipt summary |
| `SEMI-ANNUAL-BUDGET-EXECUTION-REPORT-FOR-2024.pdf` | 2024 | **OCR** — budget vs collected by revenue code |

### OCR prerequisites (Windows)
```powershell
winget install UB-Mannheim.TesseractOCR
winget install oschwartz10612.Poppler
```


## 1. Setup

In [ ]:
from __future__ import annotations

import re
import shutil
from datetime import datetime
from pathlib import Path

import pandas as pd
import pymupdf
import yaml

PROJECT_ROOT = Path("..").resolve()
import sys
sys.path.insert(0, str(PROJECT_ROOT))
from src.revenue_pdf_parser import parse_financial_statement_year
from src.revenue_ocr_parser import parse_budget_execution_rows
RAW_TEXT_DIR = PROJECT_ROOT / "data" / "raw_text"
RAW_TEXT_DIR.mkdir(parents=True, exist_ok=True)
DOCS_ROOT = (
    PROJECT_ROOT
    / "csc_4792_dataset_docs-20260911T121615Z-1-001"
    / "csc_4792_dataset_docs"
    / "Budgets & Financial Statement"
)
SCHEMAS_PATH = PROJECT_ROOT / "schemas" / "csv_schemas.yaml"
OUTPUT_DIR = PROJECT_ROOT / "data" / "processed"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TABLE_NAME = "db-unza26-csc4792-chipata_revenue_streams"
COUNCIL_NAME = "Chipata City Council"
CSV_SEP = "|"
EXTRACTED_AT = datetime.now().strftime("%Y-%m-%dT%H:%M:%S")

with open(SCHEMAS_PATH, encoding="utf-8") as f:
    SCHEMAS = yaml.safe_load(f)

COLUMNS = [c["name"] for c in SCHEMAS["tables"][TABLE_NAME]["columns"]]
print(f"Documents folder: {DOCS_ROOT}")
print(f"Output: {OUTPUT_DIR / (TABLE_NAME + '.csv')}")

In [ ]:
!pip install pymupdf

## 2. Helper functions

In [ ]:
def parse_amount(value: str | int | float | None) -> float | None:
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return None
    if isinstance(value, (int, float)):
        return round(float(value), 2)
    cleaned = re.sub(r"[^0-9.]", "", str(value))
    if not cleaned or cleaned == ".":
        return None
    try:
        return round(float(cleaned), 2)
    except ValueError:
        return None


def collection_rate(budgeted: float | None, collected: float | None) -> float | None:
    if budgeted in (None, 0) or collected is None:
        return None
    return round((collected / budgeted) * 100, 2)


def classify_revenue(description: str) -> str:
    desc = description.lower()
    rules = [
        ("market_fees", ["market fee", "market fees", "bus station fee"]),
        ("property_rates", ["residential rate", "commercial rate", "industrial rate", "property rate", "site rent"]),
        ("business_licence", ["licence", "license", "liquor", "occupancy", "retail licence", "wholesale"]),
        ("building_permit", ["permit", "fire certificate", "burial permit", "health permit"]),
        ("levy", ["levy", "personal levy", "business levy", "pole levy", "bicycle levy", "sand levy"]),
        ("fine_penalty", ["penalty", "penalties", "fine"]),
        ("rental_income", ["rental", "lease of council"]),
        ("lgef_transfer", ["local government equalisation", "lgef"]),
        ("cdf_transfer", ["constituency development fund", "cdf funding", "cdf transfer"]),
        ("grant", ["grant in lieu", "other grant", "sector grant", "roads grant", "health grant"]),
    ]
    for source_type, keywords in rules:
        if any(k in desc for k in keywords):
            return source_type
    if "fee" in desc or "charge" in desc:
        return "other"
    return "other"


def make_row(
    record_id: str,
    revenue_year: int,
    description: str,
    source_document: str,
    extraction_method: str,
    collected: float | None = None,
    budgeted: float | None = None,
    reporting_period: str = "annual",
    source_page: int | None = None,
    source_type: str | None = None,
    notes: str = "",
) -> dict:
    return {
        "record_id": record_id,
        "council_name": COUNCIL_NAME,
        "revenue_year": revenue_year,
        "revenue_source_type": source_type or classify_revenue(description),
        "revenue_description": description,
        "budgeted_amount_zmw": budgeted,
        "collected_amount_zmw": collected,
        "collection_rate_percent": collection_rate(budgeted, collected),
        "reporting_period": reporting_period,
        "source_document": source_document,
        "source_page": source_page,
        "extraction_method": extraction_method,
        "extracted_at": EXTRACTED_AT,
        "notes": notes,
    }


def rows_to_dataframe(rows: list[dict]) -> pd.DataFrame:
    df = pd.DataFrame(rows)
    for col in COLUMNS:
        if col not in df.columns:
            df[col] = None
    return df[COLUMNS]


def extract_pdf_text(pdf_path: Path, page_start: int = 1, page_end: int | None = None) -> str:
    """Extract native text from a PDF using PyMuPDF (works when PDF has a text layer)."""
    with pymupdf.open(pdf_path) as doc:
        end = page_end or doc.page_count
        chunks = []
        for i in range(page_start - 1, min(end, doc.page_count)):
            chunks.append(f"=== PAGE {i + 1} ===\n{doc[i].get_text()}")
        return "\n".join(chunks)


def pdf_needs_ocr(pdf_path: Path, min_chars: int = 100) -> bool:
    """Return True if the PDF appears to be a scan (almost no extractable text)."""
    with pymupdf.open(pdf_path) as doc:
        sample = "".join(doc[i].get_text() for i in range(min(3, doc.page_count)))
    return len(sample.strip()) < min_chars


def save_raw_text(pdf_path: Path, text: str) -> Path:
    out = RAW_TEXT_DIR / f"{pdf_path.stem}.txt"
    out.write_text(text, encoding="utf-8")
    return out

## 2b. PDF text extraction and OCR

Two methods are used depending on the PDF:

| Method | When | Library |
|--------|------|---------|
| **Native text** | PDF has a selectable text layer | `pymupdf` |
| **OCR** | PDF is a scanned image | `pdf2image` + `pytesseract` + Tesseract/Poppler |

In [ ]:
# --- OCR configuration (Windows) ---
DEFAULT_TESSERACT = Path(r"C:\Program Files\Tesseract-OCR\tesseract.exe")
DEFAULT_POPPLER_ROOT = Path(
    r"C:\Users\Windows\AppData\Local\Microsoft\WinGet\Packages"
)


def find_tesseract() -> Path | None:
    found = shutil.which("tesseract")
    if found:
        return Path(found)
    return DEFAULT_TESSERACT if DEFAULT_TESSERACT.exists() else None


def find_poppler_bin() -> Path | None:
    found = shutil.which("pdftoppm")
    if found:
        return Path(found).parent
    if DEFAULT_POPPLER_ROOT.exists():
        for pkg in sorted(DEFAULT_POPPLER_ROOT.glob("oschwartz10612.Poppler_*")):
            for sub in pkg.glob("poppler-*/Library/bin"):
                if (sub / "pdftoppm.exe").exists():
                    return sub
    return None


def ocr_pdf(pdf_path: Path, dpi: int = 250, max_pages: int | None = None) -> str:
    """Convert PDF pages to images, then run Tesseract OCR."""
    from pdf2image import convert_from_path
    import pytesseract

    poppler = find_poppler_bin()
    tesseract = find_tesseract()
    if poppler is None:
        raise RuntimeError("Install Poppler: winget install oschwartz10612.Poppler")
    if tesseract is None:
        raise RuntimeError("Install Tesseract: winget install UB-Mannheim.TesseractOCR")

    pytesseract.pytesseract.tesseract_cmd = str(tesseract)
    images = convert_from_path(
        str(pdf_path), dpi=dpi, poppler_path=str(poppler), last_page=max_pages
    )
    return "\n\n".join(
        f"=== PAGE {i + 1} ===\n{pytesseract.image_to_string(img)}"
        for i, img in enumerate(images)
    )


def get_document_text(pdf_path: Path, use_ocr: bool = False) -> tuple[str, str]:
    """Try native text first; fall back to OCR when requested."""
    text = extract_pdf_text(pdf_path)
    if len(text.strip()) > 100:
        return text, "native_text"
    if use_ocr:
        return ocr_pdf(pdf_path), "ocr"
    return text, "ocr_needed"


print(f"Tesseract: {find_tesseract()}")
print(f"Poppler:   {find_poppler_bin()}")

In [ ]:
# --- Parser for OCR budget execution tables (2024 semi-annual report) ---
OCR_AMOUNT_RE = re.compile(r"[\($]?(\d{1,3}(?:,\d{3})+|\d+)\.\d{2}[\)]?")
OCR_CODE_RE = re.compile(r"^[\[\(]?(\d{5,6})")
OCR_CODE_PREFIXES = ("151", "152", "153", "154", "155", "156", "157", "738", "130", "159")


def _ocr_parse_amounts(line: str) -> list[float]:
    out = []
    for m in OCR_AMOUNT_RE.finditer(line):
        try:
            out.append(round(float(m.group(1).replace(",", "")), 2))
        except ValueError:
            pass
    return out


def _ocr_pick_collected(amounts: list[float]) -> float:
    if len(amounts) < 2:
        return amounts[-1]
    budget = amounts[0]
    tol = max(1000.0, budget * 0.02)
    for i in range(1, len(amounts)):
        for j in range(i + 1, len(amounts)):
            if abs(budget - amounts[i] - amounts[j]) <= tol:
                return min(amounts[i], amounts[j])
    below = [a for a in amounts[1:] if a < budget]
    return max(below) if below else amounts[1]


def _ocr_clean_description(line: str) -> str:
    text = re.sub(r"^[\[\(]?\d{5,6}[\]\)\|_\[]?\s*", "", line.strip())
    text = re.sub(OCR_AMOUNT_RE, "", text)
    text = re.sub(r"[\|\[\]_]+", " ", text)
    text = re.sub(r"\s+", " ", text).strip(" :-")
    return text[:500]


def parse_budget_execution_ocr(ocr_text: str, year: int, source_file: str, start_seq: int = 1) -> list[dict]:
    """Parse revenue rows from OCR text of the semi-annual budget execution report."""
    rows: list[dict] = []
    seq = start_seq

    for chunk in ocr_text.split("=== PAGE "):
        if not chunk.strip():
            continue
        page_part, _, body = chunk.partition("===")
        try:
            page_num = int(page_part.strip())
        except ValueError:
            page_num = None
        if page_num is not None and page_num > 4:
            continue

        for line in body.splitlines():
            line = line.strip()
            if not line or "EXPENSES/EXPENDITURE" in line.upper():
                break
            m = OCR_CODE_RE.match(line)
            if not m or not m.group(1).startswith(OCR_CODE_PREFIXES):
                continue
            amounts = _ocr_parse_amounts(line)
            if len(amounts) < 2:
                continue
            desc = _ocr_clean_description(line)
            if len(desc) < 3:
                continue
            rows.append(
                make_row(
                    record_id=f"REV-{year}-{seq:03d}",
                    revenue_year=year,
                    description=desc,
                    source_document=source_file,
                    extraction_method="ocr",
                    collected=_ocr_pick_collected(amounts),
                    budgeted=amounts[0],
                    reporting_period="semi_annual",
                    source_page=page_num,
                    notes="Auto-parsed from OCR budget execution table",
                )
            )
            seq += 1
    return rows

In [ ]:
# DEMO: preview text extracted from a PDF (run this to verify extraction works)
DEMO_PDF = DOCS_ROOT / "FINANCIAL-STATEMENT.pdf"  # native text
# DEMO_PDF = DOCS_ROOT / "SEMI-ANNUAL-BUDGET-EXECUTION-REPORT-FOR-2024.pdf"  # needs OCR

if DEMO_PDF.exists():
    needs_ocr = pdf_needs_ocr(DEMO_PDF)
    print(f"File: {DEMO_PDF.name} | needs_ocr={needs_ocr}")
    demo_text, demo_method = get_document_text(DEMO_PDF, use_ocr=needs_ocr)
    save_raw_text(DEMO_PDF, demo_text)
    print(f"Method: {demo_method} | chars: {len(demo_text):,}")
    print("--- preview (first 1500 chars) ---")
    print(demo_text[:1500])

In [ ]:
def build_2024_ocr_rows(start_seq: int = 1) -> list[dict]:
    """Extract 2024 revenue rows from scanned semi-annual budget execution report."""
    pdf_path = DOCS_ROOT / "SEMI-ANNUAL-BUDGET-EXECUTION-REPORT-FOR-2024.pdf"
    if not pdf_path.exists():
        print("2024 OCR source not found")
        return []

    cache = RAW_TEXT_DIR / f"{pdf_path.stem}_ocr.txt"
    if cache.exists() and cache.stat().st_size > 500:
        ocr_text = cache.read_text(encoding="utf-8")
        print(f"Using cached OCR: {cache.name}")
    else:
        print(f"Running OCR on {pdf_path.name} (may take ~1 min)...")
        ocr_text = ocr_pdf(pdf_path)
        cache.write_text(ocr_text, encoding="utf-8")

    rows = parse_budget_execution_rows(
        ocr_text, year=2024, source_file=pdf_path.name,
        make_row=make_row, start_seq=start_seq,
    )
    print(f"Parsed {len(rows)} OCR rows for 2024")
    return rows

## 3. Build rows from PDF extraction (2022, 2023, 2025)

These functions call `parse_financial_statement_year()` in `src/revenue_pdf_parser.py`. **No Kwacha amounts are written in the notebook** — the parser opens each PDF, reads the note pages, and extracts figures from the text.

In [ ]:
FS_2023 = DOCS_ROOT / "FINANCIAL-STATEMENT.pdf"
FS_2022 = DOCS_ROOT / "FINANCIAL-STATEMENTS-FOR-THE-YEAR-ENDED-31TH.pdf"
FS_2025 = DOCS_ROOT / "1.-Chipata-City-Council-Bi-Annual-2025-Draft.pdf"

for p in [FS_2023, FS_2022, FS_2025]:
    print(f"{'OK' if p.exists() else 'MISSING'}: {p.name}")

In [ ]:
# Show native PDF text from 2023 financial statement (page 23 = local taxes note)
if FS_2023.exists():
    page23_text = extract_pdf_text(FS_2023, page_start=23, page_end=23)
    save_raw_text(FS_2023, extract_pdf_text(FS_2023, page_start=21, page_end=34))
    print("Page 23 extract (source for 2023 local tax rows):")
    print(page23_text[:1200])

In [ ]:
def build_2023_revenue_rows() -> list[dict]:
    """Extract 2023 revenue from FINANCIAL-STATEMENT.pdf — amounts parsed from PDF text."""
    pdf = DOCS_ROOT / "FINANCIAL-STATEMENT.pdf"
    if not pdf.exists():
        print(f"Missing: {pdf.name}")
        return []
    rows = parse_financial_statement_year(
        pdf, report_year=2023, source_file=pdf.name, make_row=make_row,
    )
    print(f"Parsed {len(rows)} rows for 2023 from {pdf.name}")
    return rows


def build_2022_revenue_rows(start_seq: int = 1) -> list[dict]:
    """Extract 2022 revenue from FINANCIAL-STATEMENTS-FOR-THE-YEAR-ENDED-31TH.pdf."""
    pdf = DOCS_ROOT / "FINANCIAL-STATEMENTS-FOR-THE-YEAR-ENDED-31TH.pdf"
    if not pdf.exists():
        print(f"Missing: {pdf.name}")
        return []
    rows = parse_financial_statement_year(
        pdf, report_year=2022, source_file=pdf.name, make_row=make_row, seq_start=start_seq,
    )
    print(f"Parsed {len(rows)} rows for 2022 from {pdf.name}")
    return rows


def build_2025_semi_annual_rows(start_seq: int = 1) -> list[dict]:
    """Extract Jan–Jun 2025 cash receipts from bi-annual draft (page 8)."""
    pdf = DOCS_ROOT / "1.-Chipata-City-Council-Bi-Annual-2025-Draft.pdf"
    if not pdf.exists():
        print(f"Missing: {pdf.name}")
        return []
    rows = parse_financial_statement_year(
        pdf, report_year=2025, source_file=pdf.name, make_row=make_row, seq_start=start_seq,
    )
    print(f"Parsed {len(rows)} rows for 2025 from {pdf.name}")
    return rows

## 4. Build and inspect dataset

In [ ]:
all_rows = []
all_rows.extend(build_2023_revenue_rows())
all_rows.extend(build_2022_revenue_rows())
all_rows.extend(build_2024_ocr_rows())      # automated OCR extraction
all_rows.extend(build_2025_semi_annual_rows())

revenue_df = rows_to_dataframe(all_rows)
print(f"Total rows: {len(revenue_df)}")
print(f"Years covered: {sorted(revenue_df['revenue_year'].dropna().unique())}")
revenue_df.groupby(["revenue_year", "revenue_source_type"]).size().unstack(fill_value=0)

In [ ]:
revenue_df.head(15)

In [ ]:
# Rows with both budget and collected amounts (2023 variance note)
revenue_df[revenue_df["budgeted_amount_zmw"].notna()][
    ["record_id", "revenue_year", "revenue_description", "budgeted_amount_zmw", "collected_amount_zmw", "collection_rate_percent"]
]

## 5. Validation

In [ ]:
allowed_types = {
    c["allowed_values"]
    for c in SCHEMAS["tables"][TABLE_NAME]["columns"]
    if c["name"] == "revenue_source_type"
}.pop()

assert list(revenue_df.columns) == COLUMNS, "Column order mismatch"
assert revenue_df["record_id"].is_unique, "Duplicate record_id values"
assert revenue_df["council_name"].eq(COUNCIL_NAME).all()
assert revenue_df["revenue_source_type"].isin(allowed_types).all(), "Invalid revenue_source_type"
assert revenue_df["revenue_description"].str.len().le(500).all()

# Round-trip pipe separator test
test_path = OUTPUT_DIR / "_validate_revenue.csv"
revenue_df.to_csv(test_path, sep=CSV_SEP, index=False, encoding="utf-8")
roundtrip = pd.read_csv(test_path, sep=CSV_SEP)
assert list(roundtrip.columns) == COLUMNS
test_path.unlink(missing_ok=True)

print("All validation checks passed.")

## 6. Export CSV


In [ ]:
output_path = OUTPUT_DIR / f"{TABLE_NAME}.csv"
revenue_df.to_csv(output_path, sep=CSV_SEP, index=False, encoding="utf-8")

print(f"Exported {len(revenue_df)} rows")
print(f"File: {output_path}")
print(f"Separator: '{CSV_SEP}'")
print()
print("Preview:")
print(revenue_df[["record_id", "revenue_year", "revenue_source_type", "revenue_description", "collected_amount_zmw"]].head(8).to_string(index=False))

## 7. Summary statistics 

In [ ]:
summary = (
    revenue_df.groupby("revenue_year")
    .agg(
        rows=("record_id", "count"),
        total_collected_zmw=("collected_amount_zmw", "sum"),
        sources=("revenue_source_type", "nunique"),
    )
    .reset_index()
)
summary

## 8. Limitations

Chipata's financial notes use multi-column tables. PyMuPDF often returns labels and amounts in the wrong order. The parser uses heuristics (regex, page splits, OCR cleanup like r,594,400 -> 1,594,400), so some rows can still be mis-paired even though amounts are extracted from the document, not typed in.

## C. CDF PROJECTS DATA EXTRACTION SCRIPTS

## CSC 4792 — Chipata City Council CDF Projects Dataset

**Course:** CSC 4792 Data Mining and Warehousing (2025/26)  
**Institution:** University of Zambia  
**Council:** Chipata City Council   
**Constituencies:** Chipata Central, Luangeni  
**Dataset file:** `outputs/db-unza26-csc4792-chipata_city_council_cdf_projects.csv`  
**Format:** CSV with pipe (`|`) separator  

---

## 1. Business Understanding (CRISP-DM)

**Problem:** Chipata City Council publishes CDF (Constituency Development Fund) project information across multiple official PDF documents. These documents are unstructured and difficult to analyse directly.

**Objective:** Build a structured, evidence-based dataset where each row is one CDF project and every value can be traced to an official source document.

**Scope:** CDF community projects only (approved, proposed, and not prioritized), 2022–2025.

**Data ethics rule:** No values are invented. If a field is missing in the source PDF, it remains blank in the CSV.


## 2. Methodology Overview (CRISP-DM) 

| Phase | Activity in this project | Notebook section |
|-------|-------------------------|------------------|
| Business Understanding | Define CDF tracking problem and dataset scope | Section 1 |
| Data Understanding | Identify sources, assess PDF quality, inspect outputs | Sections 6–7 |
| Data Preparation | Transcribe, structure, integrate, export CSV | Sections 8–9 |
| Exploratory analysis | Summary statistics and sample filters | Sections 10–12 |

**Data collection approach:** All source PDFs were **downloaded from the official council document portal** and saved locally in `csc_4792_dataset_docs/` before any extraction. This notebook documents how those local files were processed into a structured CSV.


## 3. Data Collection — Official Portal Downloads

### Step 1 — Download from official portal
Official Chipata City Council CDF documents were downloaded from the council document portal and placed in:

`csc_4792_dataset_docs/CDF Approved Project/`

### Step 2 — Keep raw files unchanged
Downloaded PDFs were stored without modification to preserve evidence and reproducibility.

### Step 3 — Extract locally
Because many PDFs are scanned images, data was extracted locally using Python (PyMuPDF) and manual transcription from rendered page images. No live website scraping was performed at extraction time.

## 4. Dataset Schema (24 Columns)

| Column | Description |
|--------|-------------|
| `record_id` | Unique identifier |
| `council_name` | Chipata City Council |
| `constituency` | Chipata Central or Luangeni |
| `cdf_year` | 2022–2025 |
| `project_name` | Short title from source |
| `project_description` | Full description from source |
| `project_sector` | Health, Education, Roads, etc. |
| `ward` | Ward name |
| `engineer_estimate_zmw` | Amount in ZMW (blank if not in source) |
| `approval_status` | Approved / Not Approved / Not Prioritized / Unknown |
| `status_justification` | Reason when not approved |
| `source_document` | Original PDF filename |
| `source_page` | Page number in PDF |
| `source_document_category` | Document type |
| `notes` | Transcription and quality notes |

**Categories:** `approved_projects_table`, `approval_notification`, `wdc_proposed`, `disapproved_list`, `cdf_progress_report`, `approved_community_projects`


## 5. Source Documents Used (527 Records)

| Source PDF | Records | Constituency | Year(s) |
|------------|---------|--------------|---------|
| CHIPATA-CENTRAL-2022.pdf | 29 | Chipata Central | 2022 |
| 2022-LAUNGENI.pdf (pages 4–5) | 20 | Luangeni | 2022 |
| CHIPATA-CENTRAL-2023.pdf | 23 | Chipata Central | 2023 |
| luangeni-2023.pdf (2023 pages) | 27 | Luangeni | 2023 |
| CHIPATA-CENTRAL-2024.pdf | 12 | Chipata Central | 2024 |
| luangeni-2023.pdf (2024 pages) | 23 | Luangeni | 2024 |
| 2024_CDF1.pdf | 35 | Both | 2024 |
| 2024-WDC-PROPOSED-COMMUNITY-PROJECTS-CHIPATA.pdf | 87 | Chipata Central | 2024 |
| 2024-WDC-PROPOSED-COMMUNITY-PROJECTS-LUANGENI.pdf | 74 | Luangeni | 2024 |
| APPROVED-COMMUNITY-PROJECTS-CHIPATA-CENTRAL.pdf | 15 | Chipata Central | 2025 |
| APPROVED-COMMUNITY-PROJECTS-2025-LUANGENI.pdf | 12 | Luangeni | 2025 |
| 2025-WARD-DEVELOPMENT-COMMITTEES-PROPOSED.pdf | 73 | Chipata Central | 2025 |
| 2025-DISAPPROVED-PROJECTS-WITH-JUSTIFICATIONS.pdf | 97 | Both | 2025 |

In [ ]:
from pathlib import Path

# Resolve project root whether kernel cwd is project root or notebooks/
PROJECT_ROOT = Path.cwd().resolve()
for candidate in [PROJECT_ROOT, PROJECT_ROOT.parent, PROJECT_ROOT.parent.parent]:
    if (candidate / "csc_4792_dataset_docs").exists():
        PROJECT_ROOT = candidate
        break

CDF_SOURCE_DIR = PROJECT_ROOT / "csc_4792_dataset_docs" / "CDF Approved Project"

pdf_files = sorted(CDF_SOURCE_DIR.glob("*.pdf"))
print("Official portal PDFs in local folder:", len(pdf_files))
for pdf in pdf_files:
    print(" -", pdf.name)

## 6. Data Understanding — PDF Quality Assessment

**Purpose (CRISP-DM):** Determine whether PDF pages contain selectable text or are scanned images.

**Method:** PyMuPDF `page.get_text()` character count per page.

**Finding:** Most CDF PDFs are scanned; manual transcription from rendered PNGs is required.

In [ ]:
import pymupdf

sample_pdf = CDF_SOURCE_DIR / "APPROVED-COMMUNITY-PROJECTS-CHIPATA-CENTRAL.pdf"
doc = pymupdf.open(sample_pdf)

print("Sample file:", sample_pdf.name)
print("Pages:", doc.page_count)
print("-" * 50)
for i in range(doc.page_count):
    n = len(doc[i].get_text("text").strip())
    kind = "scanned" if n == 0 else "has text"
    print(f"Page {i+1}: {n} characters -> {kind}")
doc.close()


## 7. Data Extraction Pipeline

| Step | Action | Method |
|------|--------|--------|
| 1 | Download PDFs from official portal | Manual download to local folder |
| 2 | Inventory files | `Path.glob("*.pdf")` |
| 3 | Assess page quality | PyMuPDF text check |
| 4 | Render pages to PNG | `get_pixmap(dpi=300)` → `data/discovery/` |
| 5 | Transcribe rows | Manual reading of PNG tables |
| 6 | Structure records | `make_record()` / `wdc_rows()` in Python modules |
| 7 | Integrate and export | `build_dataset()` → pipe-separated CSV |

**Design decisions:** no deduplication across documents; preserve source spelling; blank missing values.

In [ ]:
import pymupdf

DISCOVERY_DIR = PROJECT_ROOT / "data" / "discovery"
DISCOVERY_DIR.mkdir(parents=True, exist_ok=True)

render_pdf = CDF_SOURCE_DIR / "CHIPATA-CENTRAL-2022.pdf"
doc = pymupdf.open(render_pdf)
for i in range(doc.page_count):
    out = DISCOVERY_DIR / f"{render_pdf.stem}__p{i+1:02d}.png"
    if not out.exists():
        doc[i].get_pixmap(dpi=300).save(str(out))
    print("Saved:", out.name)
doc.close()

### 7.1 Example structured record from one transcribed PDF row

In [ ]:
import sys
sys.path.insert(0, str(PROJECT_ROOT / "src" / "extraction"))
from build_cdf_dataset import make_record

example = make_record(
    record_id="CCC-2025-001",
    constituency="Chipata Central",
    cdf_year=2025,
    source_row_number=1,
    project_name="Industrial Boreholes",
    project_description="Construction of Industrial Boreholes at Kabulonga in Kanjala, Mchini in Dilika and Chimwemwe in Kapata",
    project_sector="Health",
    ward="All",
    project_site_location="Kabulonga (Magazine), Mchini and Chimwemwe",
    engineer_estimate_zmw=8500000.00,
    approval_status="Approved",
    source_document_category="approved_projects_table",
    source_document="APPROVED-COMMUNITY-PROJECTS-CHIPATA-CENTRAL.pdf",
    source_page=2,
)
print("record_id:", example["record_id"])
print("source_document:", example["source_document"])
print("engineer_estimate_zmw:", example["engineer_estimate_zmw"])

## 8. Data Preparation — Build and Export Final CSV

Integrates all transcribed sources into one 527-row dataset and exports the assignment CSV.

In [ ]:
import sys
import pandas as pd

sys.path.insert(0, str(PROJECT_ROOT / "src" / "extraction"))
from build_cdf_dataset import build_dataset

OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RAW_CSV = PROJECT_ROOT / "data" / "raw" / "combined_raw_cdf_projects.csv"
RAW_CSV.parent.mkdir(parents=True, exist_ok=True)
FINAL_CSV = OUTPUT_DIR / "db-unza26-csc4792-chipata_city_council_cdf_projects.csv"

df = build_dataset()
df.to_csv(RAW_CSV, sep="|", index=False)

try:
    df.to_csv(FINAL_CSV, sep="|", index=False)
    print("Saved:", FINAL_CSV)
except PermissionError:
    print("Note: close the CSV in Excel, then re-run this cell.")
    print("Dataset saved to:", RAW_CSV)

print("Shape:", df.shape)
print("Years:", sorted(df["cdf_year"].unique()))
print("Constituencies:\n", df["constituency"].value_counts())


## 9. Load and Inspect Final Dataset


In [ ]:
# df is built in the previous cell; reload from raw CSV if running this cell alone
try:
    df
except NameError:
    import pandas as pd
    df = pd.read_csv(PROJECT_ROOT / "data" / "raw" / "combined_raw_cdf_projects.csv", sep="|")

print("Shape:", df.shape)
display(df.head(10))
print("By year and constituency:")
print(df.groupby(["cdf_year", "constituency"]).size())


## 10. Data Quality Validation

In [ ]:
cc25 = df[
    (df["constituency"] == "Chipata Central")
    & (df["cdf_year"] == 2025)
    & (df["source_document_category"] == "approved_projects_table")
]
total = round(cc25["engineer_estimate_zmw"].sum(), 2)
print("CC 2025 approved records:", len(cc25))
print("Total ZMW:", total)
print("Expected: 20916106.93")
print("Match:", total == 20916106.93)
print("\nBy source category:")
print(df["source_document_category"].value_counts())
print("\nMissing amounts:", df["engineer_estimate_zmw"].isna().sum(), "of", len(df))


## 11. Sample Exploratory Filters

In [ ]:
health = df[(df["approval_status"] == "Approved") & (df["project_sector"] == "Health")]
print("Approved Health projects:", len(health))
display(health[["record_id", "constituency", "cdf_year", "project_name", "ward"]].head(10))


## 12. Limitations

1. **Scanned PDFs:** Portal downloads are mostly image scans; data was manually transcribed from rendered PNGs.
2. **Missing amounts:** WDC and disapproved lists often omit costs — field left blank.
3. **Duplicate appearances:** Same project may appear in multiple documents; separate rows kept for provenance.
4. **Portal snapshot:** Data reflects documents available at download time.
5. **Scope:** CDF projects only — not budgets, LGEF, or IDPs.
6. **Excluded files:** Blank scans and duplicate documents omitted.
7. **Transcription risk:** Minor human error possible; mitigated by source page references.

## 13. Reproducibility

From the project root folder:

```powershell
cd src/extraction
python build_cdf_dataset.py
```

Or run all code cells in this notebook in order. Close the CSV in Excel before regenerating.

## D. AUDIT SUMMARY DATA EXTRACTION SCRIPTS

##  Chipata City Council Audit Summary Dataset

**Course:** CSC 4792 Data Mining and Warehousing (2025/26)
**Institution:** University of Zambia
**Council:** Chipata City Council (Project Team #5)
**Component:** Audit (Auditor General opinions + internal audit findings)
**Dataset file:** `db-unza26-csc4792-chipata_audit_summary.csv`
**Format:** CSV with pipe (`|`) separator

---

## 1. Business Understanding (CRISP-DM)

**Problem:** Chipata City Council's audit history is spread across scanned PDF opinion letters from the Office of the Auditor General and an internally-produced semi-annual audit table, none of which are machine-readable as-is.

**Objective:** Build a structured, evidence-based dataset — one row per audit report — where every value can be traced back to a specific source document and page, following the team's shared `chipata_audit_summary` schema.

## 2. Methodology Overview (CRISP-DM — Lectures 3 & 4)

| Phase | Activity in this notebook | Section |
|-------|---------------------------|---------|
| Business Understanding | Define the audit tracking problem and dataset scope | Section 1 |
| Data Understanding | Inspect source files, assess whether OCR is needed | Sections 4–6 |
| Data Preparation | OCR extraction, manual transcription, schema mapping, export | Sections 7–9 |
| Evaluation | Row-count and null-value sanity checks | Section 10 |

**Data collection approach:** All source files (2 PDFs, 1 screenshot, 2 report-cover images) were downloaded from the shared team Google Drive folder as a single zip and uploaded directly into this Colab session — no live scraping was performed at extraction time.

## 3. Data Collection — Source Files

The zip contains 5 files under `Audit Reports/`:

| File | Type | Content |
|------|------|---------|
| `INDEPENDENT-AUDIT-OPINION-2023.pdf` | Scanned PDF | AG's independent audit opinion, FY ended 31 Dec 2023 |
| `audit-opinion-2024.pdf` | Scanned PDF | AG's independent audit opinion, FY ended 31 Dec 2024 |
| `Screenshot_26-9-2025_16118...jpg` | Screenshot | Council's own Semi-Annual Internal Audit Report, Jan–Jun 2025 |
| `LOCAL_GOVERNMENT-REPORT-2022_page-00011.jpg` | Cover image | Cover page only of the national multi-council Local Authorities audit report (Chipata's actual section not yet obtained) |
| `CDF_REPORT-2022_page-0001...jpg` | Cover image | Cover page only of the national multi-council CDF audit report (Chipata's actual section not yet obtained) |

Raw files are kept unmodified in `audit_reports/` after extraction, to preserve evidence and reproducibility.

In [ ]:
# Install the libraries this notebook depends on:
#  - pymupdf (imported as `fitz`): renders PDF pages to images
#  - pytesseract: Python wrapper around the Tesseract OCR engine
#  - pandas: builds and exports the structured dataset
!pip install pymupdf pytesseract pandas -q

# Tesseract itself is a system binary (not a Python package), so it's installed
# separately via apt-get. Without this, pytesseract has nothing to call.
!apt-get install -y tesseract-ocr -q

### 3.1 Upload the source zip

Select the `Audit_Reports...zip` file from your local machine when prompted. Only run this cell once per session — re-running it after a successful upload can create a duplicate file with `(1)` appended to the name.

In [ ]:
from google.colab import files

# Opens a file picker in the browser. `uploaded` is a dict of
# {filename: file_bytes} for whatever you select.
uploaded = files.upload()  # select your Audit_Reports zip when prompted

In [ ]:
import zipfile, os

# Grab whatever filename Colab actually saved (avoids hardcoding a name
# that might not match if Colab appended a suffix like "(1)").
zip_name = list(uploaded.keys())[0]

# Sanity check before trying to open it, so a bad upload fails with a
# clear message instead of a raw FileNotFoundError traceback.
if not os.path.exists(zip_name):
    raise FileNotFoundError(
        f"'{zip_name}' wasn't saved properly — re-run the upload cell above, "
        "once, and wait for it to finish before running this cell."
    )

with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall('audit_reports')

# List everything that came out of the zip, to confirm the expected 5 files
# are present before moving on to extraction.
for root, dirs, fnames in os.walk('audit_reports'):
    for f in fnames:
        print(os.path.join(root, f))

## 4. Data Understanding — Are These Files Text or Scans?

**Purpose (CRISP-DM):** Before extracting anything, check whether each source file has a real, selectable text layer or is a scanned image with no text underneath. This determines whether plain text extraction works or OCR is required.

**Finding:**
- `INDEPENDENT-AUDIT-OPINION-2023.pdf` has a *fake* text layer — a previous, low-quality OCR pass left garbled text (e.g. `"CHIPA]A CI?'Y COUNCIL"`) that `pdfplumber`/`fitz.get_text()` would return as-is. Re-OCRing at higher resolution gives a clean result instead.
- `audit-opinion-2024.pdf` has **no text layer at all** — every page returns 0 characters from `get_text()`. It's a pure scanned image.
- The internal audit screenshot is a low-resolution (768×542) JPEG screenshot of a printed table — OCR on it is unreliable, so its 8 rows were transcribed by direct visual reading and only spot-checked against OCR output.

This is why the extraction below uses image-rendering + Tesseract OCR for the two PDFs, rather than a simple text-extraction call.

In [ ]:
import fitz  # this is the import name for the pymupdf package
import pytesseract
from PIL import Image
import io

def ocr_pdf(path):
    """Render every page of a PDF to an image, then run Tesseract OCR on it.

    This is needed instead of a plain text-extraction call because both
    source PDFs are scans (see Section 4): one has no text layer at all,
    and the other has a garbled fake one from a bad prior OCR pass.
    """
    doc = fitz.open(path)
    pages_text = []
    for page in doc:
        # dpi=200 balances OCR accuracy against processing time; going much
        # higher gives diminishing returns for typed report text like this.
        pix = page.get_pixmap(dpi=200)
        img = Image.open(io.BytesIO(pix.tobytes("png")))
        pages_text.append(pytesseract.image_to_string(img))
    # Join pages with a visible marker so it's obvious in the printed output
    # which page any given line of text came from.
    return "\n--- PAGE BREAK ---\n".join(pages_text)

text_2023 = ocr_pdf("audit_reports/Audit Reports/INDEPENDENT-AUDIT-OPINION-2023.pdf")
text_2024 = ocr_pdf("audit_reports/Audit Reports/audit-opinion-2024.pdf")

# Always eyeball a slice of OCR output before trusting it in your dataset —
# OCR can silently misread numbers and names.
print(text_2023[:1000])  # eyeball it to check OCR quality before trusting it

### 4.1 The internal audit screenshot

This file is a JPEG screenshot (not a PDF), and at 768×542 it's too low-resolution for Tesseract to read reliably at native size. Upscaling it 3x before OCR improves — but doesn't fully fix — recognition, so the output here is used as a cross-check against a manual reading of the table rather than trusted outright.

In [ ]:
img = Image.open("audit_reports/Audit Reports/Screenshot_26-9-2025_16118_-768x542.jpeg.jpg")

# Upscale 3x with LANCZOS resampling (a high-quality resize filter) before
# OCR — Tesseract performs noticeably better on larger text.
img_big = img.resize((img.width*3, img.height*3), Image.LANCZOS)

# --psm 6 tells Tesseract to treat the image as a single uniform block of
# text, which suits a table better than its default page-segmentation mode.
print(pytesseract.image_to_string(img_big, config='--psm 6'))

# NOTE: output here is still noisy due to the source screenshot's low
# resolution. The 8 rows used in the dataset below were transcribed by
# directly viewing the image and cross-checked against this OCR pass —
# not copied verbatim from it.

## 5. Dataset Schema (21 Columns)

Matches the team's shared `chipata_audit_summary` schema exactly — same column names and order, so this file merges cleanly with the other 7 tables in the group's Kaggle dataset.

| Column | Description |
|--------|-------------|
| `record_id` | Unique identifier (primary key) |
| `council_name` | Chipata City Council |
| `audit_report_year` | Calendar year the audit report was issued/received |
| `financial_year_ended` | Financial period-end date the audit covers |
| `auditor` | Issuing body (AG or council's own Internal Audit Unit) |
| `audit_opinion_type` | unqualified / qualified / adverse / disclaimer / unable_to_determine |
| `opinion_summary` | Paraphrase of the audit opinion paragraph |
| `key_audit_matter` | Most significant matter(s) identified |
| `cdf_receipts_zmw`, `cdf_payments_zmw`, `cdf_closing_balance_zmw` | CDF statement figures |
| `lgef_receipts_zmw`, `lgef_payments_zmw`, `lgef_closing_balance_zmw` | LGEF statement figures |
| `material_misstatement` | True / False / not_stated |
| `compliance_issues` | Non-compliance or internal control weaknesses noted |
| `source_document`, `source_page` | Provenance |
| `extraction_method` | native_text / ocr / table_parser / manual |
| `extracted_at` | Timestamp the row was created |
| `notes` | Clarifications, assumptions, and known gaps |

**Two schema-fit gaps found while mapping this data in** (documented per-row in `notes`, not silently forced):
1. `audit_opinion_type`'s enum has no value for an internal audit report, which doesn't issue an AG-style opinion — `unable_to_determine` was used as the closest fit.
2. The companion `chipata_council_documents` table's `document_type` enum similarly has no `internal_audit_report` value — `other` was used there instead.

## 6. Data Preparation — Build and Export the Final CSV

Structures the OCR/transcription output above into the 3 rows required by the schema (2023 AG opinion, 2024 AG opinion, Jan–Jun 2025 internal audit), then exports the assignment CSV.

In [ ]:
import pandas as pd

# Column order matches the team schema exactly — this matters because it's
# what makes this CSV merge cleanly with teammates' tables later.
columns = ["record_id","council_name","audit_report_year","financial_year_ended","auditor",
"audit_opinion_type","opinion_summary","key_audit_matter","cdf_receipts_zmw","cdf_payments_zmw",
"cdf_closing_balance_zmw","lgef_receipts_zmw","lgef_payments_zmw","lgef_closing_balance_zmw",
"material_misstatement","compliance_issues","source_document","source_page",
"extraction_method","extracted_at","notes"]

# One dictionary per row. Every unknown value is the literal string "NA"
# (per the schema's null-value rule), never a blank or 0 — a 0 would wrongly
# imply "the council received zero kwacha" when the truth is the figure
# simply wasn't in the source document available to us.
rows = [
    {
        "record_id": "AUD-2023-001",
        "council_name": "Chipata City Council",
        "audit_report_year": 2025,
        "financial_year_ended": "2023-12-31",
        "auditor": "Office of the Auditor General - Zambia",
        "audit_opinion_type": "unqualified",
        "opinion_summary": "Financial statements for FY2023 present fairly, in all material respects, cash receipts and payments and balances in accordance with Cash Basis IPSAS and the Local Authorities Accounting Policies (LAAPs); no key audit matters identified.",
        "key_audit_matter": "NA",
        "cdf_receipts_zmw": "NA",
        "cdf_payments_zmw": "NA",
        "cdf_closing_balance_zmw": "NA",
        "lgef_receipts_zmw": "NA",
        "lgef_payments_zmw": "NA",
        "lgef_closing_balance_zmw": "NA",
        "material_misstatement": "False",
        "compliance_issues": "Council prepared cash-basis statements (Cash Receipts and Payments, Budget vs Actual, LGEF, CDF) under Cash Basis IPSAS/LAAPs rather than accrual-basis Statement of Comprehensive Income and Financial Position referenced under Section 45 of the Local Government Act No. 2 of 2019.",
        "source_document": "INDEPENDENT-AUDIT-OPINION-2023.pdf",
        "source_page": 1,
        "extraction_method": "ocr",
        "extracted_at": "2026-09-14T07:10:00",
        "notes": "Only the 4-page opinion letter was in the source zip (no statement annexes), so CDF/LGEF ZMW figures are NA. audit_report_year is based on the Town Clerk's date-received stamp (11 Sep 2025) since the AG's own issuance date was illegible in the scan."
    },
    {
        "record_id": "AUD-2024-001",
        "council_name": "Chipata City Council",
        "audit_report_year": 2025,
        "financial_year_ended": "2024-12-31",
        "auditor": "Office of the Auditor General - Zambia",
        "audit_opinion_type": "unqualified",
        "opinion_summary": "Financial statements for FY2024 audited, covering Cash Receipts and Payments, Budget vs Actual, LGEF, CDF, Zambia Devolution Support Programme (ZDSP), and Sector Grant (Devolved Functions) statements.",
        "key_audit_matter": "Council's going concern noted as dependent on government policy decisions.",
        "cdf_receipts_zmw": "NA",
        "cdf_payments_zmw": "NA",
        "cdf_closing_balance_zmw": "NA",
        "lgef_receipts_zmw": "NA",
        "lgef_payments_zmw": "NA",
        "lgef_closing_balance_zmw": "NA",
        "material_misstatement": "False",
        "compliance_issues": "NA",
        "source_document": "audit-opinion-2024.pdf",
        "source_page": 1,
        "extraction_method": "ocr",
        "extracted_at": "2026-09-14T07:10:00",
        "notes": "Only the 4-page opinion letter was in the source zip (no statement annexes), so CDF/LGEF ZMW figures are NA. audit_report_year based on Town Clerk's date-received stamp (28 Oct 2025)."
    },
    {
        "record_id": "AUD-2025-INT-001",
        "council_name": "Chipata City Council",
        "audit_report_year": 2025,
        "financial_year_ended": "2025-06-30",
        "auditor": "Chipata City Council - Internal Audit Unit",
        "audit_opinion_type": "unable_to_determine",
        "opinion_summary": "Semi-Annual Internal Audit Report (Jan-Jun 2025) covering governance, budget execution, revenue, accounting controls, stores, procurement, payroll, and risk management. Internal audit reports of this kind do not issue a formal AG-style opinion.",
        "key_audit_matter": "Failure to regulate/control Jere market revenue collection; unmaintained stores records; irregularities in contractual staff pay-grade structure; inconsistent payment voucher formats.",
        "cdf_receipts_zmw": "NA",
        "cdf_payments_zmw": "NA",
        "cdf_closing_balance_zmw": "NA",
        "lgef_receipts_zmw": "NA",
        "lgef_payments_zmw": "NA",
        "lgef_closing_balance_zmw": "NA",
        "material_misstatement": "not_stated",
        "compliance_issues": "Governance: failure to include coopted members in audit committee; Income - Own Source Revenue: failure to regulate Jere market; Stores management: failure to maintain stores records; Procurement: gazebo tent/chair (K1,950) paid for but not delivered; Payroll: irregularities in contractual pay-grade structure; Risk management: inconsistent payment voucher formats. No findings raised for Budget Execution or Accounting procedures and controls.",
        "source_document": "Screenshot_26-9-2025_16118_-768x542.jpeg.jpg",
        "source_page": 1,
        "extraction_method": "manual",
        "extracted_at": "2026-09-14T07:10:00",
        "notes": "SCHEMA GAP: this is an internal audit report, not an AG opinion, so audit_opinion_type doesn't cleanly apply - used 'unable_to_determine' as closest enum value. Full per-finding detail (recommendation, management action taken) kept in a separate supplementary CSV since this schema holds one compliance_issues text field per record."
    },
]

# Build the table: pandas matches each dict's keys to the `columns` list,
# so the export below always has predictable, schema-correct column order.
df_audit = pd.DataFrame(rows, columns=columns)
df_audit  # displayed for a visual check before exporting

## 7. Data Quality Validation

A quick sanity check before exporting — confirms the row count is what's expected and that the required (non-`NA`) columns are actually populated.

In [ ]:
print("Shape:", df_audit.shape)  # expect (3, 21): 3 audit reports, 21 schema columns

# Required columns (per the schema, marked Req=Yes) should never be "NA" —
# this catches accidental blanks before they reach the shared dataset.
required_cols = ["record_id","council_name","audit_report_year","financial_year_ended",
                  "auditor","audit_opinion_type","opinion_summary","material_misstatement",
                  "source_document","extraction_method","extracted_at"]
for col in required_cols:
    n_missing = (df_audit[col] == "NA").sum()
    print(f"{col}: {n_missing} rows marked NA (should be 0)")

print("\nExtraction methods used:")
print(df_audit["extraction_method"].value_counts())

## 8. Export the Final CSV

Saved with a pipe separator and the required naming convention, then downloaded to your local machine.

In [ ]:
df_audit.to_csv("db-unza26-csc4792-chipata_audit_summary.csv", sep="|", index=False, encoding="utf-8")

from google.colab import files
files.download("db-unza26-csc4792-chipata_audit_summary.csv")

## 9. Limitations

1. **Two source files are cover pages only:** the 2022 CDF and Local Authorities national audit reports were only available as their cover images, not the full multi-council PDF — Chipata's specific section (with real findings and figures) has not yet been extracted. These are excluded from this CSV pending the full documents.
2. **No financial statement annexes available:** the two AG opinion letters obtained are 4-page cover letters only, without the underlying Statement of Cash Receipts and Payments annexes — so all `cdf_*_zmw` and `lgef_*_zmw` columns are `NA`, not because the council didn't report them, but because this notebook's source files didn't include them.
3. **OCR quality varies by source:** the 2024 opinion PDF is a clean scan and OCRs reliably; the 2023 PDF's embedded (pre-existing) text layer is garbled from a prior bad OCR pass and was disregarded in favour of re-OCRing the rendered image.
4. **Screenshot resolution:** the internal audit table (768×542 JPEG) is too low-resolution for fully reliable OCR; its 8 rows were transcribed by direct visual reading, cross-checked against OCR output.
5. **Schema enum gaps:** `audit_opinion_type` and (in the companion `chipata_council_documents` table) `document_type` have no value that cleanly fits an internal audit report — documented per-row in `notes` rather than forced into a misleading category.
6. **Scope:** this table covers audit opinions and audit findings only — not budgets, CDF projects, revenue, or IDPs, which are separate tables owned by teammates.

## 10. Reproducibility

Run all cells in this notebook in order, from a fresh Colab runtime:

1. Run Section 3's install cell.
2. Run the upload cell and select the `Audit_Reports...zip` file — only once per session.
3. Run the remaining cells top to bottom.

If `zipfile.ZipFile` raises `FileNotFoundError`, restart the runtime (`Runtime > Restart session`) and re-upload once, waiting for the upload to fully finish before running the next cell.

## E. FINANCIAL STATEMENTS SCRIPTS

### Output table: db-unza26-csc4792-chipata_financial_statements.csv
### Source folder: Budgets & Financial Statement
### Council: Chipata City Council

This notebook documents the extraction, parsing, and structuring of financial statement data published by Chipata City Council. Folders excluded from this table: Adverts, CDF Approved Project, Council Minutes, Stakeholders Meeting, Service Delivery Charter and Zambia Devolution Support Program.

### Configure paths and extraction scope

Setting the Google Drive root folder, subfolder keywords (Budget, Financial statement, Audit report), output filenames, council name, and OCR thresholds. Only PDF and image files whose folder path matches these keywords are queued — this prevents Adverts and CDF notification PDFs in the same archive from being processed.

In [ ]:
DRIVE_FOLDER = "/content/drive/MyDrive/csc_4792_dataset_docs"

FOLDER_KEYWORDS = [
    "budget",
    "financial statement",
    "audit report",
]

OUTPUT_XLSX = "db-unza26-csc4792-chipata_financial_statements.xlsx"
OUTPUT_CSV  = "db-unza26-csc4792-chipata_financial_statements.csv"

COUNCIL_NAME = "Chipata City Council"
CURRENCY = "ZMW"

MIN_NATIVE_CHARS = 80
OCR_DPI = 200

print("DRIVE_FOLDER:", DRIVE_FOLDER)
print("FOLDER_KEYWORDS:", FOLDER_KEYWORDS)

## Setup

Mounting Google Drive and installing dependencies for native PDF text extraction (pdfplumber), scanned-page OCR (Tesseract, Poppler, pdf2image), and structured export (pandas, openpyxl).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!apt-get -qq install -y tesseract-ocr poppler-utils > /dev/null
!pip -q install pdfplumber pymupdf pytesseract pdf2image pandas openpyxl pillow

import os, re, json
from datetime import datetime
from pathlib import Path

import pandas as pd
import pdfplumber
import pytesseract
from pdf2image import convert_from_path
from PIL import Image

print("Environment ready.")

## Confirm archive structure

Listing the directory tree under DRIVE_FOLDER before extraction to verify that **Budgets & Financial Statement** and **Audit Reports** are accessible. In our archive the documents sit one level inside csc_4792_dataset_docs/; the recursive file search handles this nested layout.

In [ ]:
def browse_folder(root: str, max_depth: int = 3):
    root = Path(root)
    if not root.exists():
        print(f"Path not found: {root}")
        return
    print(f"Archive root: {root}\n")
    for dirpath, dirnames, filenames in os.walk(root):
        depth = len(Path(dirpath).relative_to(root).parts)
        if depth > max_depth:
            dirnames.clear()
            continue
        indent = "  " * depth
        print(f"{indent}{Path(dirpath).name}/")
        for fn in sorted(filenames)[:8]:
            print(f"{indent}  {fn}")
        if len(filenames) > 8:
            print(f"{indent}  ... +{len(filenames)-8} more files")

browse_folder(DRIVE_FOLDER)

## Build extraction and parsing functions

Defining the pipeline mapped to the 20-column chipata_financial_statements schema.

Statement page detection requires both a recognisable IPSAS title (Statement of Cash Receipts and Payments, Budget vs Actual, LGEF, CDF, debt/arrears, bi-annual/semi-annual reports) and evidence of numeric table content (TOTAL RECEIPTS, comma-formatted kwacha amounts). Audit opinion cover pages that only *mention* statements in narrative prose are skipped.

Native PDF text is attempted first; pages with fewer than 80 characters trigger Tesseract OCR at 200 DPI. Parsed rows are classified by statement_type, und_type, and line_category (receipt, payment, balance). Letterhead patterns (telephone, P.O. Box, auditor text) are filtered at parse time.

In [ ]:
FIN_COLS = [
    "record_id", "council_name", "financial_year", "period_start", "period_end",
    "statement_type", "fund_type", "line_item", "line_category",
    "amount_zmw", "budget_amount_zmw", "actual_amount_zmw",
    "opening_balance_zmw", "closing_balance_zmw", "currency",
    "source_document", "source_page", "extraction_method", "extracted_at", "notes",
]

STATEMENT_PATTERNS = [
    (r"statement\s+of\s+cash\s+receipts\s+and\s+payments\s+for\s+the\s+local\s+government\s+equali[sz]ation\s+fund",
     "lgef_statement", "lgef"),
    (r"statement\s+of\s+cash\s+receipts\s+and\s+payments\s+for\s+the\s+constituency\s+development\s+fund",
     "cdf_statement", "cdf"),
    (r"statement\s+of\s+comparison\s+of\s+budget\s+and\s+actual",
     "budget_vs_actual", "general"),
    (r"statement\s+of\s+cash\s+receipts\s+and\s+payments",
     "statement_of_cash_receipts_and_payments", "general"),
    (r"debt\s+and\s+arrears|debt\s+and\s+areas|debt\s+.*\s+arrears|debtors?\s+and\s+creditors?",
     "debt_and_arrears", "general"),
    (r"bi[-\s]?annual\s+(financial\s+statement|budget)|semi[-\s]?annual\s+budget\s+(execution|performance)|budget\s+(execution|performance)\s+report",
     "bi_annual_performance", "general"),
]

STATEMENT_PAGE_PATTERNS = [
    r"statement\s+of\s+cash\s+receipts\s+and\s+payments\s+for\s+the\s+year",
    r"statement\s+of\s+cash\s+receipts\s+and\s+payments\s+for\s+the\s+local\s+government",
    r"statement\s+of\s+cash\s+receipts\s+and\s+payments\s+for\s+the\s+constituency",
    r"statement\s+of\s+comparison\s+of\s+budget\s+and\s+actual\s+amounts",
    r"statement\s+of\s+cash\s+receipts\s+and\s+payments",
    r"debt\s+and\s+arrears|debt\s+and\s+areas|schedule\s+of\s+debt|debtors?\s+and\s+creditors?",
    r"bi[-\s]?annual\s+(financial\s+statement|budget\s+(performance|execution))",
    r"semi[-\s]?annual\s+budget\s+(execution|performance)",
    r"budget\s+execution\s+report|budget\s+performance\s+report",
]

JUNK_DESC_RE = re.compile(
    r"telephone|tel\.|p\.?\s*o\.?\s*box|stand\s*no|stand\s*plot|"
    r"auditor\s*general|office\s+of\s+the\s+auditor|fax|e-?mail|"
    r"website|www\.|highway|umodzi|cnic\s+centre|accordance\s+with|"
    r"internal\s+control|laaps|ipsas|report\s+on\s+other\s+legal|"
    r"acting\s+auditor|signature|position\:|general\s+auditor",
    re.I,
)

AMOUNT_RE = re.compile(r"(?<!\d)(?:\([\d,]+\)|[\d,]+(?:\.\d+)?)(?!\d)")
YEAR_ENDED_RE = re.compile(
    r"year\s+ended\s+(?:31(?:st|th|rd|nd)?\s+)?(?:december|dec\.?\s*)?(\d{4})", re.I
)
YEAR_IN_NAME_RE = re.compile(r"(20\d{2})")

_record_counter = 0


def next_record_id(financial_year):
    global _record_counter
    _record_counter += 1
    return f"FIN-{financial_year}-{_record_counter:03d}"


def parse_amount(s):
    s = s.replace(",", "").strip()
    neg = s.startswith("(") and s.endswith(")")
    s = s.strip("()")
    try:
        v = float(s)
        return -v if neg else v
    except ValueError:
        return None


def infer_financial_year(text, filename):
    m = YEAR_ENDED_RE.search(text)
    if m:
        return int(m.group(1))
    years = YEAR_IN_NAME_RE.findall(filename)
    return int(years[-1]) if years else datetime.now().year


def infer_period_end(financial_year):
    return f"{financial_year}-12-31"


def detect_statement(text):
    low = re.sub(r"\s+", " ", text.lower())
    for pattern, stype, fund in STATEMENT_PATTERNS:
        if re.search(pattern, low):
            return stype, fund
    return "other", "consolidated"


def classify_line_category(line, section):
    low = line.lower()
    if "opening balance" in low or "cash at beginning" in low or "cash at the start" in low:
        return "balance"
    if "closing balance" in low or "cash at end" in low or "cash at the end" in low:
        return "balance"
    if "total" in low:
        return "balance"
    if "transfer" in low:
        return "transfer"
    if section == "RECEIPTS":
        return "receipt"
    if section == "PAYMENTS":
        return "payment"
    return "expenditure"


def is_header_or_noise(line):
    up = line.upper().strip()
    noise = {
        "NOTE", "KWACHA", "K", "RECEIPTS", "PAYMENTS", "PAGE",
        "SIGNATURE", "POSITION", "PERFORMANCE", "VARIANCE",
    }
    if up in noise or len(up) < 3:
        return True
    if re.fullmatch(r"[\d\s%./-]+", up):
        return True
    if re.search(r"^\d{4}\s+\d{4}$", up):
        return True
    return False


def is_likely_financial_row(desc, amounts):
    if JUNK_DESC_RE.search(desc):
        return False
    if re.search(
        r"statement of|financial statement|for the year ended|chipata city council|"
        r"report of|table of contents|independent auditor|prepared the statement",
        desc,
        re.I,
    ):
        return False
    if re.fullmatch(r"[\d\s./%-]+", desc):
        return False
    amt = parse_amount(amounts[0])
    if amt is None:
        return False
    has_comma = "," in amounts[0]
    low = desc.lower()
    is_balance = any(x in low for x in ("cash at", "opening", "closing", "balance"))
    if abs(amt) < 100 and not has_comma and not is_balance:
        return False
    if abs(amt) < 10 and not is_balance:
        return False
    return True


def parse_standard_statement(text, meta):
    rows = []
    section = None
    fy = meta["financial_year"]
    stype, fund = detect_statement(text)

    for raw in text.splitlines():
        line = raw.strip()
        if not line:
            continue
        up = re.sub(r"\s+", " ", line.upper())
        if up in ("RECEIPTS", "PAYMENTS", "RECEIPT", "PAYMENT"):
            section = "RECEIPTS" if "RECEIPT" in up else "PAYMENTS"
            continue
        if is_header_or_noise(line):
            continue
        amounts = AMOUNT_RE.findall(line)
        if not amounts:
            continue
        desc = AMOUNT_RE.sub("", line).strip(" .:-_#")
        if len(desc) < 2 or not re.search(r"[A-Za-z]{2,}", desc):
            continue
        if not is_likely_financial_row(desc, amounts):
            continue

        amt = parse_amount(amounts[0])
        if amt is None:
            continue

        cat = classify_line_category(line, section or "")
        opening = closing = budget = actual = None
        low = desc.lower()
        if "beginning" in low or "opening" in low:
            opening = amt
        elif "end" in low and "year" in low:
            closing = amt

        rows.append({
            "record_id": next_record_id(fy),
            "council_name": COUNCIL_NAME,
            "financial_year": fy,
            "period_start": f"{fy}-01-01",
            "period_end": infer_period_end(fy),
            "statement_type": stype,
            "fund_type": fund,
            "line_item": desc[:500],
            "line_category": cat,
            "amount_zmw": amt,
            "budget_amount_zmw": budget,
            "actual_amount_zmw": actual,
            "opening_balance_zmw": opening,
            "closing_balance_zmw": closing,
            "currency": CURRENCY,
            "source_document": meta["source_document"],
            "source_page": meta["source_page"],
            "extraction_method": meta["extraction_method"],
            "extracted_at": meta["extracted_at"],
            "notes": "",
        })
    return rows


def parse_budget_vs_actual(text, meta):
    rows = []
    fy = meta["financial_year"]
    section = None

    for raw in text.splitlines():
        line = raw.strip()
        if not line:
            continue
        up = re.sub(r"\s+", " ", line.upper())
        if up in ("RECEIPTS", "PAYMENTS"):
            section = up
            continue
        if is_header_or_noise(line):
            continue
        amounts = AMOUNT_RE.findall(line)
        if len(amounts) < 2:
            continue
        desc = AMOUNT_RE.sub("", line).strip(" .:-_%")
        if len(desc) < 2:
            continue
        if not is_likely_financial_row(desc, amounts):
            continue

        parsed = [parse_amount(a) for a in amounts if parse_amount(a) is not None]
        if len(parsed) < 2:
            continue
        budget = parsed[-2] if len(parsed) >= 2 else None
        actual = parsed[-1]

        rows.append({
            "record_id": next_record_id(fy),
            "council_name": COUNCIL_NAME,
            "financial_year": fy,
            "period_start": f"{fy}-01-01",
            "period_end": infer_period_end(fy),
            "statement_type": "budget_vs_actual",
            "fund_type": "general",
            "line_item": desc[:500],
            "line_category": classify_line_category(line, section or ""),
            "amount_zmw": actual,
            "budget_amount_zmw": budget,
            "actual_amount_zmw": actual,
            "opening_balance_zmw": None,
            "closing_balance_zmw": None,
            "currency": CURRENCY,
            "source_document": meta["source_document"],
            "source_page": meta["source_page"],
            "extraction_method": meta["extraction_method"],
            "extracted_at": meta["extracted_at"],
            "notes": "Budget vs actual; amounts inferred from column order.",
        })
    return rows


def page_has_statement_table(text):
    low = text.lower()
    if "total receipts" in low or "total payments" in low:
        return True
    comma_lines = sum(
        1 for line in text.splitlines() if re.search(r"\d{1,3},\d{3}", line)
    )
    if comma_lines >= 2 and ("receipts" in low or "payments" in low):
        return True
    if "budget" in low and "actual" in low and comma_lines >= 2:
        return True
    if re.search(r"debt|arrears|areas|debtor|creditor|owing|payable", low):
        amount_lines = sum(
            1 for line in text.splitlines()
            if AMOUNT_RE.findall(line)
            and len(AMOUNT_RE.sub("", line).strip()) > 2
            and not JUNK_DESC_RE.search(line)
        )
        if amount_lines >= 2:
            return True
    if re.search(r"bi[-\s]?annual|semi[-\s]?annual|budget execution|budget performance", low):
        if comma_lines >= 2:
            return True
    return False


def page_is_statement_page(text):
    low = re.sub(r"\s+", " ", text.lower())
    has_title = any(re.search(p, low) for p in STATEMENT_PAGE_PATTERNS)
    has_table = page_has_statement_table(text)

    narrative_only = (
        re.search(r"prepared\s+the\s+statement\s+of|in\s+accordance\s+with|"
                  r"auditor['']?s\s+report|report\s+on\s+other\s+legal", low)
        and not has_table
    )
    if narrative_only:
        return False

    if has_title and has_table:
        return True
    if has_table and (
        "total receipts" in low or "total payments" in low
        or ("equalisation fund" in low and "receipts" in low)
        or ("constituency development fund" in low and "receipts" in low)
        or re.search(r"debt|arrears|areas|budget execution|budget performance", low)
    ):
        return True
    return False


def extract_rows_from_page(text, meta):
    if not page_is_statement_page(text):
        return []
    low = text.lower()
    if "comparison of budget and actual" in low or "budget and actual amounts" in low:
        return parse_budget_vs_actual(text, meta)
    return parse_standard_statement(text, meta)


def ocr_image(img):
    return pytesseract.image_to_string(img)


def extract_pdf_pages(filepath):
    pages = []
    fname = Path(filepath).name

    try:
        with pdfplumber.open(filepath) as pdf:
            for i, page in enumerate(pdf.pages, start=1):
                native = (page.extract_text() or "").strip()
                if len(native) >= MIN_NATIVE_CHARS:
                    pages.append((i, native, "native_text"))
                else:
                    pages.append((i, None, "pending_ocr"))
    except Exception as e:
        print(f"  pdfplumber error on {fname}: {e}")
        return []

    if any(p[2] == "pending_ocr" for p in pages):
        try:
            images = convert_from_path(filepath, dpi=OCR_DPI)
            for idx, (pn, txt, meth) in enumerate(pages):
                if meth == "pending_ocr" and pn <= len(images):
                    pages[idx] = (pn, ocr_image(images[pn - 1]), "ocr")
        except Exception as e:
            print(f"  OCR error on {fname}: {e}")
            pages = [(pn, txt or "", "ocr_failed" if meth == "pending_ocr" else meth)
                     for pn, txt, meth in pages]

    return [(pn, txt or "", meth) for pn, txt, meth in pages]


def extract_image(filepath):
    try:
        img = Image.open(filepath)
        return [(1, ocr_image(img), "ocr")]
    except Exception as e:
        print(f"  Image OCR error: {e}")
        return []


def folder_matches(path_str):
    low = path_str.lower()
    return any(kw in low for kw in FOLDER_KEYWORDS)


def discover_files(root):
    root = Path(root)
    exts = {".pdf", ".jpg", ".jpeg", ".png"}
    files = []
    for fp in root.rglob("*"):
        if fp.suffix.lower() not in exts:
            continue
        rel = str(fp.relative_to(root))
        if folder_matches(rel):
            files.append(fp)
    return sorted(files)


def process_file(filepath):
    fname = filepath.name
    ext = filepath.suffix.lower()
    extracted_at = datetime.now().strftime("%Y-%m-%dT%H:%M:%S")

    if ext == ".pdf":
        page_data = extract_pdf_pages(str(filepath))
    else:
        page_data = extract_image(str(filepath))

    all_rows = []
    methods_used = set()
    sample_text = ""

    for page_num, text, method in page_data:
        methods_used.add(method)
        if not sample_text and text:
            sample_text = text
        fy = infer_financial_year(text or sample_text, fname)
        meta = {
            "source_document": fname,
            "source_page": page_num,
            "extraction_method": method if method != "pending_ocr" else "ocr",
            "extracted_at": extracted_at,
            "financial_year": fy,
        }
        rows = extract_rows_from_page(text, meta)
        all_rows.extend(rows)

    method_summary = ",".join(sorted(methods_used)) if methods_used else "none"
    if not page_data:
        status, notes = "EMPTY", "Could not read file"
    elif not all_rows:
        status, notes = "EMPTY", "No financial statement tables detected"
    elif any(m in method_summary for m in ("ocr", "ocr_failed")):
        status = "PARTIAL" if all_rows else "EMPTY"
        notes = f"OCR applied; {len(all_rows)} rows extracted"
    else:
        status = "OK" if len(all_rows) >= 3 else "PARTIAL"
        notes = f"{len(all_rows)} rows extracted"

    report = {
        "source_document": fname,
        "folder_path": str(filepath.parent.name),
        "file_format": ext.lstrip("."),
        "page_count": len(page_data),
        "rows_extracted": len(all_rows),
        "extraction_method": method_summary,
        "status": status,
        "notes": notes,
    }
    return all_rows, report

print("Extraction functions loaded.")

## Run batch extraction across all source documents

Discovering PDF/image files matching the folder keywords and processing each document page-by-page. Each source receives a status in the Extraction Report: **OK** (reliable native-text tables), **PARTIAL** (OCR used and/or sparse extraction), or **EMPTY** (no tabular statements — expected for audit cover JPGs and opinion-only PDFs).

In [ ]:
global _record_counter
_record_counter = 0

target_files = discover_files(DRIVE_FOLDER)
print(f"Documents queued: {len(target_files)}\n")
for f in target_files:
    print(f"  {f.relative_to(DRIVE_FOLDER)}")

all_records = []
reports = []

for i, fp in enumerate(target_files, 1):
    print(f"\n[{i}/{len(target_files)}] {fp.name}")
    rows, report = process_file(fp)
    all_records.extend(rows)
    reports.append(report)
    print(f"  {report['status']}: {report['rows_extracted']} rows ({report['extraction_method']})")

df_fin = pd.DataFrame(all_records, columns=FIN_COLS)
df_report = pd.DataFrame(reports)

print("\n" + "=" * 60)
print(f"Total records: {len(df_fin)}")
print(f"OK: {(df_report['status']=='OK').sum()} | PARTIAL: {(df_report['status']=='PARTIAL').sum()} | EMPTY: {(df_report['status']=='EMPTY').sum()}")
df_fin.head(10)

## Clean and preprocess extracted records

Raw extraction yielded 339 rows. Scanned documents — especially Financial-statement-2024.pdf — produced OCR artefacts such as note numbers parsed as amounts (Local taxes = 36 instead of millions) and calendar years stored as amount_zmw.

Preprocessing steps applied before export:

1. Drop letterhead/narrative junk in line_item (telephone, P.O. Box, auditor text).
2. Drop standalone Note rows and amounts equal to calendar years (2019–2030).
3. Drop receipt/payment rows below K10,000 (implausible for council-scale line items).
4. Drop total/subtotal rows below K100,000.

Common OCR spelling fixes (e.g. PAIANENTS → PAYMENTS, oflhe → of the) are applied via regex corrections before filtering. Rows removed per rule are logged in the Preprocessing Log sheet.

In [ ]:
CORRECTIONS = {
    r"\bPAIANENTS\b": "PAYMENTS",
    r"\boflhe\b": "of the",
    r"\bofthe\b": "of the",
    r"\bvear\b": "year",
    r"\bCrsh\b": "Cash",
}

def normalize_text(s):
    for pat, repl in CORRECTIONS.items():
        s = re.sub(pat, repl, s, flags=re.I)
    # collapse "Incre as e" -> "Increase" style splits: single/double-letter
    # fragments separated by spaces, glued back together
    s = re.sub(r"\b([A-Za-z]{2,3})\s+([a-z]{1,2})\s+([a-z]{1,3})\b", r"\1\2\3", s)
    return re.sub(r"\s+", " ", s).strip()

def preprocess_financial_statements(df):
    """Return cleaned DataFrame and a log of rows removed by rule."""
    df = df.copy()
    logs = []

    df["line_item"] = df["line_item"].astype(str).apply(normalize_text)

    def drop_mask(mask, rule_name):
        n = int(mask.sum())
        if n:
            logs.append({"rule": rule_name, "rows_removed": n})
        return ~mask

    keep = pd.Series(True, index=df.index)

    # Rule 1: letterhead / narrative junk in line_item
    junk_pat = re.compile(
        r"telephone|p\.?\s*o\.?\s*box|stand\s*no|auditor\s*general|"
        r"office\s+of\s+the\s+auditor|accordance\s+with|signature",
        re.I,
    )
    keep &= drop_mask(df["line_item"].astype(str).str.contains(junk_pat, na=False), "letterhead_or_narrative")

    # Rule 2: standalone Note rows and year-valued amounts
    note_row = df["line_item"].astype(str).str.strip().str.lower().eq("note")
    year_amount = df["amount_zmw"].apply(
        lambda x: isinstance(x, (int, float)) and 2019 <= x <= 2030 and float(x) == int(float(x))
    )
    keep &= drop_mask(note_row | year_amount, "note_reference_or_year_amount")

    # Rule 3: receipt/payment lines with implausible small amounts (OCR column/note bleed)
    small_receipt_payment = (
        df["line_category"].isin(["receipt", "payment", "expenditure"])
        & (df["amount_zmw"].abs() < 10_000)
    )
    keep &= drop_mask(small_receipt_payment, "implausible_small_receipt_payment")

    # Rule 4: totals/subtotals with implausibly small amounts
    total_like = df["line_item"].astype(str).str.contains(
        r"total\s+receipt|total\s+payment|sub[-\s]?total", re.I, na=False
    )
    small_total = total_like & (df["amount_zmw"].abs() < 100_000)
    keep &= drop_mask(small_total, "implausible_small_total")

    cleaned = df.loc[keep].reset_index(drop=True)
    log_df = pd.DataFrame(logs)
    return cleaned, log_df


df_fin_raw = df_fin.copy()
df_fin, df_preprocess_log = preprocess_financial_statements(df_fin_raw)

print(f"Raw extracted records:    {len(df_fin_raw)}")
print(f"After preprocessing:      {len(df_fin)}")
print(f"Records removed:          {len(df_fin_raw) - len(df_fin)}")
print()
display(df_preprocess_log)
df_fin.head(10)

## Export final dataset

Writing preprocessed records to the Kaggle-ready CSV (UTF-8, pipe | separator, filename db-unza26-csc4792-chipata_financial_statements.csv) and an Excel workbook with sheets: chipata_financial_statements, Extraction Report, and Preprocessing Log.

In [ ]:
out_dir = Path(DRIVE_FOLDER)
xlsx_path = out_dir / OUTPUT_XLSX
csv_path = out_dir / OUTPUT_CSV

with pd.ExcelWriter(xlsx_path, engine="openpyxl") as writer:
    df_fin.to_excel(writer, sheet_name="chipata_financial_statements", index=False)
    df_report.to_excel(writer, sheet_name="Extraction Report", index=False)
    df_preprocess_log.to_excel(writer, sheet_name="Preprocessing Log", index=False)

df_fin.to_csv(csv_path, sep="|", index=False, encoding="utf-8")

print(f"Excel: {xlsx_path}")
print(f"CSV:   {csv_path}")
display(df_report)